# チュートリアル: Custom Memory + Azure AI Search

この notebook では Agent Framework の `ContextProvider` と Azure AI Search だけで custom memory layer を作ります。

狙いは次の通りです。

| 領域 | この notebook の設計 |
|---|---|
| 記憶ストア | Azure AI Search custom index |
| 検索方式 | keyword / vector / hybrid / semantic hybrid / recency / importance を選択 |
| ソート | `created_at`、`updated_at`、`importance`、`last_accessed_at` などの top-level field で実行 |
| 記憶抽出 | LLM に JSON で memory candidate を抽出させる。LLM が使えない場合は簡易 fallback |
| 記憶統合 | 近い既存 memory を Azure AI Search で探し、LLM または rule fallback で `ADD` / `UPDATE` / `DELETE` / `NONE` を判断 |
| 人間らしい記憶 | 重要度、アクセス回数、最終アクセス、期限、減衰、アーカイブを扱う |

記憶の「抽出」と「統合判断」は class として明示的に分離します。これにより、検索 schema と memory lifecycle は Azure AI Search 側で自由に設計できます。

## 1. 依存関係

この notebook は `azure-search-documents`、`azure-identity`、`openai`、`python-dotenv` を使います。Agent Framework の `ContextProvider` は、この workshop に含まれるローカル package を優先して読み込みます。

In [ ]:
# 必要に応じてコメントアウトを外して実行してください。
# %pip install -q "azure-search-documents==11.5.2" azure-identity openai python-dotenv

print("依存関係セルを確認しました。未導入の場合は上の %pip install を実行してください。")

## 2. 環境変数と設定

Azure AI Search は memory store、Azure OpenAI は memory 抽出・統合判断・embedding に使います。

In [ ]:
# pyright: reportMissingImports=false, reportUnusedImport=false, reportUnusedVariable=false

import json
import logging
import math
import os
import re
import sys
import uuid
from dataclasses import asdict, dataclass, field
from datetime import datetime, timezone
from enum import StrEnum
from pathlib import Path
from typing import Any

from dotenv import load_dotenv

load_dotenv()

LOCAL_AGENT_FRAMEWORK_PACKAGES = Path("agent-framework-python-1.4.0/python/packages")
for package_name in ["core", "openai", "azure-ai-search"]:
    package_path = LOCAL_AGENT_FRAMEWORK_PACKAGES / package_name
    if package_path.exists():
        sys.path.insert(0, str(package_path.resolve()))

# === Azure AI Search ===
AZURE_SEARCH_ENDPOINT = os.getenv("AZURE_SEARCH_ENDPOINT")
AZURE_SEARCH_API_KEY = os.getenv("AZURE_SEARCH_API_KEY")
AZURE_SEARCH_INDEX_NAME = os.getenv("CUSTOM_MEMORY_SEARCH_INDEX_NAME", "custom-memory")

# === Azure OpenAI ===
AZURE_OPENAI_ENDPOINT = os.getenv("AZURE_OPENAI_ENDPOINT")
AZURE_OPENAI_API_KEY = os.getenv("AZURE_OPENAI_API_KEY")
AZURE_OPENAI_API_VERSION = os.getenv("AZURE_OPENAI_API_VERSION", "2024-10-21")
AZURE_OPENAI_CHAT_DEPLOYMENT = os.getenv("AZURE_OPENAI_CHAT_DEPLOYMENT")
AZURE_OPENAI_EMBEDDING_DEPLOYMENT = os.getenv("AZURE_OPENAI_EMBEDDING_DEPLOYMENT_NAME")
AZURE_OPENAI_EMBEDDING_DIMS = int(os.getenv("AZURE_OPENAI_EMBEDDING_DIMS", "1536"))

# === デモ用 ID ===
DEMO_USER_ID = os.getenv("CUSTOM_MEMORY_DEMO_USER_ID", "workshop-user")
DEMO_AGENT_ID = os.getenv("CUSTOM_MEMORY_DEMO_AGENT_ID", "travel-agent")

# === 認証方式の選択 ===
# True: APIキー認証, False: DefaultAzureCredential 認証
USE_KEY_AUTH = os.getenv("USE_KEY_AUTH", "False").lower() in ("true", "1", "t")

if USE_KEY_AUTH:
    print("🔑 認証方式: APIキー認証")
else:
    os.environ.pop("AZURE_OPENAI_API_KEY", None)
    print("🔐 認証方式: DefaultAzureCredential")

# === 準備状況 ===
SEARCH_READY = bool(AZURE_SEARCH_ENDPOINT and AZURE_SEARCH_INDEX_NAME)
LLM_READY = bool(AZURE_OPENAI_ENDPOINT and AZURE_OPENAI_CHAT_DEPLOYMENT)
EMBEDDING_READY = bool(AZURE_OPENAI_ENDPOINT and AZURE_OPENAI_EMBEDDING_DEPLOYMENT)

print(f"\nSEARCH_READY: {SEARCH_READY}")
print(f"LLM_READY: {LLM_READY}")
print(f"EMBEDDING_READY: {EMBEDDING_READY}")

# === ロガー ===
custom_memory_logger = logging.getLogger("custom_memory.aisearch")
custom_memory_logger.setLevel(logging.INFO)
if not custom_memory_logger.handlers:
    _handler = logging.StreamHandler()
    _handler.setFormatter(logging.Formatter("%(asctime)s %(levelname)s %(name)s - %(message)s"))
    custom_memory_logger.addHandler(_handler)
custom_memory_logger.propagate = False

### 2.1. OpenTelemetry によるトレーサーのセット
マルチエージェントのデバッグには OpenTelemetry によるトレーサーを利用すると便利です。
この環境の Agent Framework では `setup_observability` は提供されていないため、
OpenTelemetry の `TracerProvider`（Exporter/Processor 含む）を手動でセットし、`enable_instrumentation()` で計測を有効化します。
ここではトレースの送信先として OTLP gRPC（例: Jaeger / OpenTelemetry Collector の `4317`）を使います。

Jaeger UI: [http://localhost:16686](http://localhost:16686)

In [ ]:
import os
from contextlib import nullcontext


class NoOpSpan:
    def set_attribute(self, _name, _value) -> None:
        del _name, _value
        return None


class NoOpTracer:
    def start_as_current_span(self, _span_name):
        del _span_name
        return nullcontext(NoOpSpan())


service_name = os.getenv("OTEL_SERVICE_NAME", "custom-memory-aisearch-notebook")
otlp_endpoint = os.getenv("OTEL_EXPORTER_OTLP_ENDPOINT", "http://localhost:4317")

os.environ.setdefault("OTEL_SERVICE_NAME", service_name)
os.environ.setdefault("OTEL_EXPORTER_OTLP_TRACES_ENDPOINT", otlp_endpoint)
os.environ.setdefault("OTEL_EXPORTER_OTLP_LOGS_ENDPOINT", otlp_endpoint)
os.environ.setdefault("OTEL_EXPORTER_OTLP_PROTOCOL", "grpc")

try:
    from agent_framework.observability import configure_otel_providers, get_tracer

    if not globals().get("CUSTOM_MEMORY_OTEL_CONFIGURED", False):
        configure_otel_providers(
            enable_sensitive_data=os.getenv("CUSTOM_MEMORY_TRACE_SENSITIVE_DATA", "true").casefold() in {"1", "true", "yes", "y"},
        )
        CUSTOM_MEMORY_OTEL_CONFIGURED = True
    tracer = get_tracer()
    print(f"Observability configured: service={service_name} otlp={otlp_endpoint}")
except Exception as error:
    tracer = NoOpTracer()
    CUSTOM_MEMORY_OTEL_CONFIGURED = False
    print("OpenTelemetry を初期化できませんでした。No-Op tracer を使います:", error)

## 3. メモリー スキーマとライフサイクル

この章では、会話から抽出した記憶を「あとから検索し、必要なら忘れられるデータ」として扱うための基本形を定義します。単に文章を保存するだけなら `memory` だけでも足りますが、エージェントに渡す記憶は毎回の応答品質に直接影響します。そのため、検索、絞り込み、順位付け、忘却判断に使う情報を最初からフィールドとして持たせます。

| 定義するもの | 役割 | なぜ必要か |
|---|---|---|
| `MemoryRecord` | Azure AI Search に保存する 1 件の記憶 | 本文だけでなく、重要度、状態、更新日時、参照回数を一緒に管理するため |
| `MemoryStatus` | `active`、`archived`、`deleted` の状態 | 古い記憶や削除済みの記憶を、既定検索から外すため |
| `SearchMode` | キーワード、ベクトル、ハイブリッド、セマンティックの切り替え | どの検索アルゴリズムで候補を取るかを明示するため |
| `RankingProfile` | 重要度重視、最近更新重視、ブーストなしの切り替え | 検索方式とは独立して Azure AI Search のスコアリング プロファイルを選ぶため |
| `MemorySearchOptions` | 検索時の条件をまとめる設定 | 検索方式、順位付け方針、件数、カテゴリ、古い記憶を含めるかどうかを、呼び出し側から明示するため |
| `HumanMemoryPolicy` | 想起、減衰、忘却のルール | 使われた記憶は残しやすくし、古く重要度の低い記憶は自然に見えにくくするため |

Mem0 の `payload` のような自由な JSON にすべてを閉じ込めると、Azure AI Search のフィルター、ソート、スコアリング プロファイルで扱いにくくなります。このノートブックでは、`created_at`、`updated_at`、`importance`、`retention_score`、`access_count` などをトップレベル フィールドとして設計し、Azure AI Search 側で順位付けに使える形にします。

In [ ]:
class MemoryStatus(StrEnum):
    ACTIVE = "active"
    ARCHIVED = "archived"
    DELETED = "deleted"


class SearchMode(StrEnum):
    KEYWORD = "keyword"
    VECTOR = "vector"
    HYBRID = "hybrid"
    SEMANTIC = "semantic"


class RankingProfile(StrEnum):
    NONE = "none"
    IMPORTANT = "important"
    RECENT = "recent"


class MemoryEvent(StrEnum):
    ADD = "ADD"
    UPDATE = "UPDATE"
    DELETE = "DELETE"
    NONE = "NONE"


MEMORY_PRIORITY_SCORING_PROFILE = "memory-priority"
MEMORY_RECENCY_SCORING_PROFILE = "memory-recency"


def utc_now() -> datetime:
    return datetime.now(timezone.utc)


def to_utc_datetime(value: datetime | str | None) -> datetime | None:
    if value is None:
        return None
    if isinstance(value, datetime):
        parsed = value
    else:
        try:
            parsed = datetime.fromisoformat(value.replace("Z", "+00:00"))
        except ValueError:
            return None
    if parsed.tzinfo is None:
        parsed = parsed.replace(tzinfo=timezone.utc)
    return parsed.astimezone(timezone.utc)


def dt_to_json(value: datetime | None) -> str | None:
    return value.isoformat() if value else None


def normalize_text(value: str) -> str:
    return re.sub(r"\s+", " ", value.strip()).casefold()


@dataclass
class MemoryCandidate:
    memory: str
    category: str = "general"
    importance: float = 0.5
    confidence: float = 0.7
    source: str = "conversation"
    expires_at: datetime | None = None
    metadata: dict[str, Any] = field(default_factory=dict)


@dataclass
class MemoryRecord:
    id: str
    user_id: str
    agent_id: str
    memory: str
    category: str = "general"
    summary: str = ""
    importance: float = 0.5
    confidence: float = 0.7
    status: MemoryStatus = MemoryStatus.ACTIVE
    source: str = "conversation"
    created_at: datetime = field(default_factory=utc_now)
    updated_at: datetime = field(default_factory=utc_now)
    last_accessed_at: datetime | None = None
    expires_at: datetime | None = None
    access_count: int = 0
    retention_score: float = 1.0
    metadata: dict[str, Any] = field(default_factory=dict)

    def to_search_document(self, embedding: list[float] | None = None) -> dict[str, Any]:
        document = {
            "id": self.id,
            "user_id": self.user_id,
            "agent_id": self.agent_id,
            "memory": self.memory,
            "summary": self.summary or self.memory,
            "category": self.category,
            "importance": float(self.importance),
            "confidence": float(self.confidence),
            "status": str(self.status),
            "source": self.source,
            "created_at": self.created_at,
            "updated_at": self.updated_at,
            "last_accessed_at": self.last_accessed_at,
            "expires_at": self.expires_at,
            "access_count": int(self.access_count),
            "retention_score": float(self.retention_score),
            "metadata_json": json.dumps(self.metadata, ensure_ascii=False),
        }
        if embedding is not None:
            document["embedding"] = embedding
        return document

    @classmethod
    def from_search_document(cls, document: dict[str, Any]) -> "MemoryRecord":
        metadata_raw = document.get("metadata_json") or "{}"
        try:
            metadata = json.loads(metadata_raw)
        except json.JSONDecodeError:
            metadata = {"raw_metadata": metadata_raw}
        status_value = str(document.get("status") or MemoryStatus.ACTIVE)
        return cls(
            id=str(document["id"]),
            user_id=str(document.get("user_id") or ""),
            agent_id=str(document.get("agent_id") or ""),
            memory=str(document.get("memory") or ""),
            category=str(document.get("category") or "general"),
            summary=str(document.get("summary") or ""),
            importance=float(document.get("importance") or 0.5),
            confidence=float(document.get("confidence") or 0.7),
            status=MemoryStatus(status_value),
            source=str(document.get("source") or "conversation"),
            created_at=to_utc_datetime(document.get("created_at")) or utc_now(),
            updated_at=to_utc_datetime(document.get("updated_at")) or utc_now(),
            last_accessed_at=to_utc_datetime(document.get("last_accessed_at")),
            expires_at=to_utc_datetime(document.get("expires_at")),
            access_count=int(document.get("access_count") or 0),
            retention_score=float(document.get("retention_score") or 1.0),
            metadata=metadata,
        )


@dataclass
class MemorySearchOptions:
    mode: SearchMode = SearchMode.HYBRID
    ranking_profile: RankingProfile = RankingProfile.IMPORTANT
    top: int = 5
    order_by: str | None = None
    category: str | None = None
    include_archived: bool = False
    min_importance: float | None = None
    min_retention_score: float | None = None
    semantic_configuration_name: str = "memory-semantic"


@dataclass
class MemorySearchResult:
    record: MemoryRecord
    score: float | None = None
    reranker_score: float | None = None


@dataclass
class ConsolidationDecision:
    event: MemoryEvent
    candidate: MemoryCandidate
    target_id: str | None = None
    merged_memory: str | None = None
    reason: str = ""


class HumanMemoryPolicy:
    def __init__(
        self,
        *,
        half_life_days: float = 45.0,
        access_boost: float = 0.04,
        archive_threshold: float = 0.22,
    ):
        self.half_life_days = half_life_days
        self.access_boost = access_boost
        self.archive_threshold = archive_threshold

    def retention_score(self, record: MemoryRecord, now: datetime | None = None) -> float:
        now = now or utc_now()
        age_days = max((now - record.updated_at).total_seconds() / 86400, 0.0)
        decay = 0.5 ** (age_days / self.half_life_days)
        access_rehearsal = min(record.access_count * self.access_boost, 0.35)
        importance_anchor = 0.35 + (0.65 * record.importance)
        if record.expires_at and record.expires_at <= now:
            return 0.0
        return max(0.0, min(1.0, decay * importance_anchor + access_rehearsal))

    def status_for(self, record: MemoryRecord, now: datetime | None = None) -> MemoryStatus:
        score = self.retention_score(record, now=now)
        if score <= self.archive_threshold:
            return MemoryStatus.ARCHIVED
        return MemoryStatus.ACTIVE

    def touch(self, record: MemoryRecord, now: datetime | None = None) -> MemoryRecord:
        now = now or utc_now()
        record.last_accessed_at = now
        record.access_count += 1
        record.retention_score = self.retention_score(record, now=now)
        if record.status != MemoryStatus.DELETED:
            record.status = self.status_for(record, now=now)
        return record

print("Memory schema と HumanMemoryPolicy を定義しました。")

### `MemorySearchOptions` の使い方

`MemorySearchOptions` は、「どの検索方式で候補を取り、どの順位付け方針で並べ、どこまでエージェントに見せるか」を表す設定オブジェクトです。記憶検索では、単に近い文章を探すだけでは不十分です。古すぎる記憶、削除済みの記憶、別カテゴリの記憶まで混ざると、エージェントが古い前提で回答してしまうためです。

基本の使い方は、検索時に `options=MemorySearchOptions(...)` を渡す形です。`mode` は候補を取る検索方式、`ranking_profile` は Azure AI Search のスコアリング プロファイルを使った順位付け方針です。たとえば `SearchMode.HYBRID` と `RankingProfile.IMPORTANT` を組み合わせると、キーワード検索とベクトル検索で候補を取り、重要度や保持スコアを順位に反映できます。

```python
options = MemorySearchOptions(
    mode=SearchMode.HYBRID,
    ranking_profile=RankingProfile.IMPORTANT,
    top=5,
    min_retention_score=0.1,
)
results = custom_memory_provider.search_memories("大阪旅行の好み", options=options)
```

主な項目は次のように読みます。

| 項目 | 使いどころ | 例 |
|---|---|---|
| `mode` | 候補を取得する検索方式を切り替える | 正確な語句なら `SearchMode.KEYWORD`、意味的類似も見たい時は `SearchMode.HYBRID` |
| `ranking_profile` | 候補集合の順位付け方針を切り替える | 長期的な制約は `RankingProfile.IMPORTANT`、直近の条件は `RankingProfile.RECENT`、比較用には `RankingProfile.NONE` |
| `top` | 返す件数を決める | コンテキストに入れる記憶は 3 から 5 件程度に抑える |
| `category` | 記憶の種類を絞る | 旅行、食事、業務設定など、用途別に検索する |
| `include_archived` | アーカイブ済みの記憶も候補に入れる | 忘却や監査の確認時だけ `True` にする |
| `min_importance` | 重要度の低い記憶を除外する | 応答に強く影響させたい時に指定する |
| `min_retention_score` | 保持スコアの低い記憶を除外する | 古く信頼しにくい記憶を既定の検索から外す |
| `order_by` | 明示的に完全な並べ替えをしたい時に使う | 監査や一覧表示で `updated_at desc` を指定する |

呼び出し側が Azure AI Search のスコアリング プロファイル名を直接指定する必要はありません。`RankingProfile.IMPORTANT` なら `memory-priority`、`RankingProfile.RECENT` なら `memory-recency` をストア側が内部で選びます。Azure AI Search の仕様上、スコアリング プロファイルは keyword だけでなく vector や hybrid でも使えますが、ブースト対象は `importance` や `updated_at` のような非ベクトル フィールドです。

## 4. 記憶抽出と統合判断

記憶の抽出と統合判断の流れを、ここでは明示的な class に分けます。

1. `MemoryExtractor`: 会話から memory candidate を抽出する
2. `MemoryConsolidator`: 新 candidate と近い既存 memory を比較し、`ADD` / `UPDATE` / `DELETE` / `NONE` を決める
3. `AzureOpenAIMemoryIntelligence`: Azure OpenAI を使う薄い JSON helper

LLM が使えない場合でも notebook の構造を観察できるように、fallback rule も持たせます。

In [ ]:
class AzureOpenAIMemoryIntelligence:
    def __init__(
        self,
        *,
        endpoint: str | None,
        api_key: str | None,
        api_version: str,
        chat_deployment: str | None,
        embedding_deployment: str | None,
        embedding_dims: int,
    ):
        self.endpoint = endpoint
        self.api_key = api_key
        self.api_version = api_version
        self.chat_deployment = chat_deployment
        self.embedding_deployment = embedding_deployment
        self.embedding_dims = embedding_dims
        self.client = None
        if endpoint and (chat_deployment or embedding_deployment):
            try:
                from openai import AzureOpenAI

                kwargs = {"azure_endpoint": endpoint, "api_version": api_version}
                if api_key:
                    kwargs["api_key"] = api_key
                self.client = AzureOpenAI(**kwargs)
            except Exception as error:
                print("AzureOpenAI client を初期化できませんでした。fallback を使います:", error)

    @property
    def can_chat(self) -> bool:
        return bool(self.client and self.chat_deployment)

    @property
    def can_embed(self) -> bool:
        return bool(self.client and self.embedding_deployment)

    def complete_json(self, *, system_prompt: str, user_prompt: str) -> dict[str, Any] | None:
        if not self.can_chat:
            return None
        response = self.client.chat.completions.create(
            model=self.chat_deployment,
            messages=[
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": user_prompt},
            ],
            response_format={"type": "json_object"},
            temperature=0.1,
        )
        content = response.choices[0].message.content or "{}"
        try:
            return json.loads(content)
        except json.JSONDecodeError:
            match = re.search(r"\{.*\}", content, flags=re.DOTALL)
            return json.loads(match.group(0)) if match else None

    def embed(self, text: str) -> list[float]:
        if not self.can_embed:
            return stable_hash_embedding(text, dimensions=self.embedding_dims)
        response = self.client.embeddings.create(model=self.embedding_deployment, input=text)
        return list(response.data[0].embedding)


def stable_hash_embedding(text: str, dimensions: int = 1536) -> list[float]:
    buckets = [0.0] * dimensions
    tokens = re.findall(r"\w+", text.casefold()) or [text]
    for token in tokens:
        token_hash = hash(token)
        buckets[abs(token_hash) % dimensions] += 1.0 if token_hash >= 0 else -1.0
    norm = math.sqrt(sum(value * value for value in buckets)) or 1.0
    return [value / norm for value in buckets]


def coerce_unit_score(value: Any, *, default: float) -> float:
    if value is None:
        return default

    if isinstance(value, (int, float)):
        score = float(value)
    elif isinstance(value, str):
        text = value.strip().casefold()
        if not text:
            return default
        label_map = {
            "very high": 0.95,
            "high": 0.85,
            "medium": 0.60,
            "mid": 0.60,
            "low": 0.35,
            "very low": 0.15,
        }
        if text in label_map:
            score = label_map[text]
        else:
            match = re.search(r"-?\d+(?:\.\d+)?", text)
            if not match:
                return default
            score = float(match.group(0))
            if "%" in text:
                score /= 100.0
    else:
        return default

    if score > 1.0 and score <= 100.0:
        score /= 100.0
    return max(0.0, min(1.0, score))


class MemoryExtractor:
    def __init__(self, intelligence: AzureOpenAIMemoryIntelligence | None = None):
        self.intelligence = intelligence

    def extract(self, messages: list[dict[str, str]]) -> list[MemoryCandidate]:
        if self.intelligence and self.intelligence.can_chat:
            extracted = self._extract_with_llm(messages)
            if extracted:
                return extracted
        return self._extract_with_rules(messages)

    def _extract_with_llm(self, messages: list[dict[str, str]]) -> list[MemoryCandidate]:
        system_prompt = """
会話から、ユーザーにとって永続的な記憶を抽出します。
重要な記憶のみを含むJSONを返します。
各記憶項目には、memory、category、importance、confidence、source、expires_at、metadataを含める必要があります。
ユーザーの言語を使用します。アシスタント専用の主張は無視します。永続的な設定、計画、制約、身元、および定期的なニーズを優先します。
後で役に立たない一時的な事実は保存しないでください。
""".strip()
        user_prompt = json.dumps({"messages": messages}, ensure_ascii=False)
        payload = self.intelligence.complete_json(system_prompt=system_prompt, user_prompt=user_prompt)
        candidates = []
        for item in (payload or {}).get("memories", []):
            if not isinstance(item, dict) or not item.get("memory"):
                continue
            candidates.append(
                MemoryCandidate(
                    memory=str(item["memory"]),
                    category=str(item.get("category") or "general"),
                    importance=coerce_unit_score(item.get("importance"), default=0.5),
                    confidence=coerce_unit_score(item.get("confidence"), default=0.7),
                    source=str(item.get("source") or "conversation"),
                    expires_at=to_utc_datetime(item.get("expires_at")),
                    metadata=item.get("metadata") if isinstance(item.get("metadata"), dict) else {},
                )
            )
        return candidates

    def _extract_with_rules(self, messages: list[dict[str, str]]) -> list[MemoryCandidate]:
        candidates: list[MemoryCandidate] = []
        durable_patterns = [
            (r"(好き|好み|優先|苦手|嫌い|必要|いつも|毎回|名前は|私は|旅行では|出張では)", "preference"),
            (r"(prefer|like|dislike|always|usually|my name is|i am|for trips|for work)", "preference"),
        ]
        for message in messages:
            if message.get("role") != "user":
                continue
            content = message.get("content", "")
            for sentence in re.split(r"(?<=[。.!?？])\s+|\n+", content):
                sentence = sentence.strip()
                if not sentence:
                    continue
                category = next((name for pattern, name in durable_patterns if re.search(pattern, sentence, re.IGNORECASE)), None)
                if category:
                    candidates.append(
                        MemoryCandidate(
                            memory=sentence,
                            category=category,
                            importance=0.65,
                            confidence=0.55,
                            source="rule-fallback",
                        )
                    )
        return candidates


class MemoryConsolidator:
    def __init__(
        self,
        intelligence: AzureOpenAIMemoryIntelligence | None = None,
        *,
        update_threshold: float = 0.78,
        none_threshold: float = 0.92,
    ):
        self.intelligence = intelligence
        self.update_threshold = update_threshold
        self.none_threshold = none_threshold

    def decide(self, candidate: MemoryCandidate, neighbors: list[MemorySearchResult]) -> ConsolidationDecision:
        if self.intelligence and self.intelligence.can_chat:
            decision = self._decide_with_llm(candidate, neighbors)
            if decision:
                return decision
        return self._decide_with_rules(candidate, neighbors)

    def _decide_with_llm(
        self,
        candidate: MemoryCandidate,
        neighbors: list[MemorySearchResult],
    ) -> ConsolidationDecision | None:
        system_prompt = """
あなたは記憶の統合エンジンです。
以下のイベントから、必ず1つだけ選択してください：ADD、UPDATE、DELETE、NONE
ADDは、候補を新しい記憶として保存することを意味します。
UPDATEは、候補を既存の記憶に統合し、対象IDを維持することを意味します。
DELETEは、候補が既存の記憶と矛盾しているか、ユーザーから削除を求められたため、既存の記憶を削除済みとしてマークすることを意味します。
NONEは、候補がすでにカバーされていることを意味します。
JSON形式でのみ、event、target_id、merged_memory、reasonを返してください。
""".strip()
        payload = {
            "candidate": asdict(candidate) | {"expires_at": dt_to_json(candidate.expires_at)},
            "neighbors": [
                {
                    "id": result.record.id,
                    "memory": result.record.memory,
                    "category": result.record.category,
                    "score": result.score,
                    "importance": result.record.importance,
                    "updated_at": dt_to_json(result.record.updated_at),
                }
                for result in neighbors
            ],
        }
        result = self.intelligence.complete_json(system_prompt=system_prompt, user_prompt=json.dumps(payload, ensure_ascii=False))
        if not result:
            return None
        try:
            event = MemoryEvent(str(result.get("event")))
        except ValueError:
            return None
        return ConsolidationDecision(
            event=event,
            candidate=candidate,
            target_id=result.get("target_id"),
            merged_memory=result.get("merged_memory") or candidate.memory,
            reason=result.get("reason") or "llm",
        )

    def _decide_with_rules(self, candidate: MemoryCandidate, neighbors: list[MemorySearchResult]) -> ConsolidationDecision:
        if not neighbors:
            return ConsolidationDecision(MemoryEvent.ADD, candidate, reason="no similar memory")
        top = neighbors[0]
        score = top.score or 0.0
        candidate_text = normalize_text(candidate.memory)
        existing_text = normalize_text(top.record.memory)
        if candidate_text == existing_text or score >= self.none_threshold:
            return ConsolidationDecision(MemoryEvent.NONE, candidate, target_id=top.record.id, reason="already covered")
        if self._looks_like_forget_request(candidate.memory):
            return ConsolidationDecision(MemoryEvent.DELETE, candidate, target_id=top.record.id, reason="forget/delete intent")
        if score >= self.update_threshold and candidate.category == top.record.category:
            merged = self._merge_text(top.record.memory, candidate.memory)
            return ConsolidationDecision(MemoryEvent.UPDATE, candidate, target_id=top.record.id, merged_memory=merged, reason="similar category")
        return ConsolidationDecision(MemoryEvent.ADD, candidate, reason="new distinct memory")

    def _looks_like_forget_request(self, text: str) -> bool:
        return bool(re.search(r"(忘れて|削除|覚えないで|取り消し|forget|delete|do not remember)", text, re.IGNORECASE))

    def _merge_text(self, existing: str, new_text: str) -> str:
        if normalize_text(new_text) in normalize_text(existing):
            return existing
        if normalize_text(existing) in normalize_text(new_text):
            return new_text
        return f"{existing} / 追加: {new_text}"

print("MemoryExtractor と MemoryConsolidator を定義しました。")

## 5. Azure AI Search メモリー ストア

記憶を Azure AI Search のドキュメントとして保存し、検索方式、フィルター、順位付け方針を `MemorySearchOptions` で切り替えます。

この実装では、`SearchMode` と `RankingProfile` を分けます。`SearchMode` は keyword / vector / hybrid / semantic のどれで候補を取るかだけを表し、`RankingProfile` は Azure AI Search のスコアリング プロファイルを使って候補をどう上位に寄せるかを表します。`RankingProfile.IMPORTANT` は `memory-priority` を内部で使い、`importance`、`retention_score`、`access_count` をブーストします。`RankingProfile.RECENT` は `memory-recency` を内部で使い、最近更新された記憶をより強く上位に寄せます。

| オプションの内容 | ストア側で起きること |
|---|---|
| `mode=SearchMode.KEYWORD` | キーワード検索だけで記憶を探す |
| `mode=SearchMode.VECTOR` | 埋め込みベクトルだけで近い記憶を探す |
| `mode=SearchMode.HYBRID` | キーワード検索とベクトル検索を組み合わせる |
| `mode=SearchMode.SEMANTIC` | ハイブリッド検索にセマンティック ランカーを加えて再順位付けする |
| `ranking_profile=RankingProfile.NONE` | Azure AI Search のスコアリング プロファイルを使わない |
| `ranking_profile=RankingProfile.IMPORTANT` | `memory-priority` を適用し、重要度、保持スコア、参照回数をまとめて順位に反映する |
| `ranking_profile=RankingProfile.RECENT` | `memory-recency` を適用し、最近更新された記憶を上位に寄せる |
| `category`、`min_importance`、`min_retention_score` | Azure AI Search のフィルター式に変換される |
| `include_archived` | アーカイブ済みの記憶を検索対象に戻す |
| `order_by` | スコアではなく完全な並べ替えが必要な場合だけ使う |

`created_at`、`updated_at`、`importance`、`retention_score`、`access_count` などはトップレベル フィールドなので、Azure AI Search のスコアリング関数、フィルター、ソートにそのまま使えます。今回の主役は、Python 側で順位を後処理することではなく、検索エンジン側に記憶の優先度を伝えることです。

### RankingProfile.IMPORTANT と RankingProfile.RECENT の有効性
検索方式を「類似度」、順位付け方針を「記憶のライフサイクル」として分離します。これにより、`SearchMode.VECTOR` や `SearchMode.HYBRID` のまま、重要度重視や最近更新重視のブーストを組み合わせられます。

| 欲しい判断 | keyword/vector だけだと弱い理由 |
|---|---|
| 最近変わった予定を優先したい | 意味的に近くても古い条件が混ざる |
| 長期的に重要な好みを優先したい | 古いが重要な記憶が新しい雑多な記憶に負ける |
| よく思い出される制約を強く扱いたい | vector 類似度には「よく使う」が入らない |
| 一時的な予定を自然に弱めたい | semantic 類似度だけでは期限や保持価値が分からない |

Azure AI Search の scoring profile は vector / hybrid query でも指定できます。ただし boost が直接かかるのはベクトル値そのものではなく、vector で得られた候補内の `importance` や `updated_at` などの非ベクトル フィールドです。

In [ ]:
# pyright: reportMissingImports=false

from datetime import timedelta

from azure.core.credentials import AzureKeyCredential
from azure.identity import DefaultAzureCredential
from azure.search.documents import SearchClient
from azure.search.documents.indexes import SearchIndexClient
from azure.search.documents.indexes.models import (
    FreshnessScoringFunction,
    FreshnessScoringParameters,
    HnswAlgorithmConfiguration,
    MagnitudeScoringFunction,
    MagnitudeScoringParameters,
    ScoringProfile,
    SearchableField,
    SearchField,
    SearchFieldDataType,
    SearchIndex,
    SemanticConfiguration,
    SemanticField,
    SemanticPrioritizedFields,
    SemanticSearch,
    SimpleField,
    TextWeights,
    VectorSearch,
    VectorSearchProfile,
)
from azure.search.documents.models import QueryType, VectorizedQuery


def create_vectorized_query(*, vector: list[float], k: int, fields: str) -> VectorizedQuery:
    attribute_map = getattr(VectorizedQuery, "_attribute_map", {}) or {}
    supported_attributes = set(attribute_map)
    kwargs: dict[str, Any] = {"vector": vector, "fields": fields}
    if "k_nearest_neighbors" in supported_attributes:
        kwargs["k_nearest_neighbors"] = k
    else:
        kwargs["k"] = k
    return VectorizedQuery(**kwargs)


class AzureAISearchMemoryStore:
    def __init__(
        self,
        *,
        endpoint: str,
        index_name: str,
        api_key: str | None,
        embedding_dims: int,
        semantic_configuration_name: str = "memory-semantic",
        vector_field_name: str = "embedding",
        auto_create: bool = True,
    ):
        self.endpoint = endpoint
        self.index_name = index_name
        self.embedding_dims = embedding_dims
        self.semantic_configuration_name = semantic_configuration_name
        self.vector_field_name = vector_field_name
        credential = AzureKeyCredential(api_key) if api_key else DefaultAzureCredential()
        self.search_client = SearchClient(endpoint=endpoint, index_name=index_name, credential=credential)
        self.index_client = SearchIndexClient(endpoint=endpoint, credential=credential)
        if auto_create:
            self.ensure_index()

    def ensure_index(self) -> None:
        existing_names = set(self.index_client.list_index_names())
        if self.index_name in existing_names:
            self._ensure_scoring_profiles()
            return

        fields = [
            SimpleField(name="id", type=SearchFieldDataType.String, key=True),
            SimpleField(name="user_id", type=SearchFieldDataType.String, filterable=True, sortable=True),
            SimpleField(name="agent_id", type=SearchFieldDataType.String, filterable=True, sortable=True),
            SearchableField(name="memory", type=SearchFieldDataType.String, searchable=True),
            SearchableField(name="summary", type=SearchFieldDataType.String, searchable=True),
            SearchableField(name="category", type=SearchFieldDataType.String, searchable=True, filterable=True, facetable=True),
            SimpleField(name="status", type=SearchFieldDataType.String, filterable=True, facetable=True),
            SimpleField(name="source", type=SearchFieldDataType.String, filterable=True, facetable=True),
            SimpleField(name="importance", type=SearchFieldDataType.Double, filterable=True, sortable=True, facetable=True),
            SimpleField(name="confidence", type=SearchFieldDataType.Double, filterable=True, sortable=True),
            SimpleField(name="retention_score", type=SearchFieldDataType.Double, filterable=True, sortable=True),
            SimpleField(name="access_count", type=SearchFieldDataType.Int32, filterable=True, sortable=True),
            SimpleField(name="created_at", type=SearchFieldDataType.DateTimeOffset, filterable=True, sortable=True),
            SimpleField(name="updated_at", type=SearchFieldDataType.DateTimeOffset, filterable=True, sortable=True),
            SimpleField(name="last_accessed_at", type=SearchFieldDataType.DateTimeOffset, filterable=True, sortable=True),
            SimpleField(name="expires_at", type=SearchFieldDataType.DateTimeOffset, filterable=True, sortable=True),
            SimpleField(name="metadata_json", type=SearchFieldDataType.String),
            SearchField(
                name=self.vector_field_name,
                type="Collection(Edm.Single)",
                searchable=True,
                vector_search_dimensions=self.embedding_dims,
                vector_search_profile_name="memory-vector-profile",
            ),
        ]
        vector_search = VectorSearch(
            profiles=[VectorSearchProfile(name="memory-vector-profile", algorithm_configuration_name="memory-hnsw")],
            algorithms=[HnswAlgorithmConfiguration(name="memory-hnsw")],
        )
        semantic_search = SemanticSearch(
            configurations=[
                SemanticConfiguration(
                    name=self.semantic_configuration_name,
                    prioritized_fields=SemanticPrioritizedFields(
                        content_fields=[SemanticField(field_name="memory"), SemanticField(field_name="summary")],
                        keywords_fields=[SemanticField(field_name="category")],
                    ),
                )
            ]
        )
        index = SearchIndex(
            name=self.index_name,
            fields=fields,
            vector_search=vector_search,
            semantic_search=semantic_search,
            scoring_profiles=self._memory_scoring_profiles(),
        )
        self.index_client.create_or_update_index(index)

    def close(self) -> None:
        self.search_client.close()
        self.index_client.close()

    def upsert(self, record: MemoryRecord, embedding: list[float]) -> None:
        response = self.search_client.merge_or_upload_documents([record.to_search_document(embedding=embedding)])
        self._raise_if_failed(response, "upsert", record.id)

    def mark_status(self, memory_id: str, status: MemoryStatus) -> None:
        response = self.search_client.merge_documents([{"id": memory_id, "status": str(status), "updated_at": utc_now()}])
        self._raise_if_failed(response, "mark_status", memory_id)

    def get(self, memory_id: str) -> MemoryRecord | None:
        try:
            document = self.search_client.get_document(key=memory_id)
        except Exception:
            return None
        return MemoryRecord.from_search_document(dict(document))

    def search(
        self,
        *,
        query: str,
        query_vector: list[float] | None,
        user_id: str,
        agent_id: str,
        options: MemorySearchOptions,
    ) -> list[MemorySearchResult]:
        filter_expression = self._filter_expression(user_id=user_id, agent_id=agent_id, options=options)
        order_by = self._order_by_for(options)
        scoring_profile = self._scoring_profile_for(options)
        params: dict[str, Any] = {
            "filter": filter_expression,
            "top": options.top,
            "select": [
                "id", "user_id", "agent_id", "memory", "summary", "category", "status", "source",
                "importance", "confidence", "retention_score", "access_count", "created_at", "updated_at",
                "last_accessed_at", "expires_at", "metadata_json",
            ],
        }

        mode = options.mode
        if mode == SearchMode.VECTOR:
            if query_vector is None:
                return []
            params["vector_queries"] = [create_vectorized_query(vector=query_vector, k=options.top, fields=self.vector_field_name)]
        elif mode in {SearchMode.HYBRID, SearchMode.SEMANTIC}:
            params["search_text"] = query or "*"
            if query_vector is not None:
                params["vector_queries"] = [create_vectorized_query(vector=query_vector, k=max(options.top, 20), fields=self.vector_field_name)]
            params["search_fields"] = ["memory", "summary", "category"]
        else:
            params["search_text"] = query or "*"
            params["search_fields"] = ["memory", "summary", "category"]

        if mode == SearchMode.SEMANTIC:
            params["query_type"] = QueryType.SEMANTIC
            params["semantic_configuration_name"] = options.semantic_configuration_name
        if scoring_profile:
            params["scoring_profile"] = scoring_profile
        if order_by:
            params["order_by"] = [order_by]

        search_results = self.search_client.search(**params)
        results = []
        for document in search_results:
            record = MemoryRecord.from_search_document(dict(document))
            score = document.get("@search.score")
            reranker_score = document.get("@search.reranker_score")
            results.append(MemorySearchResult(record=record, score=score, reranker_score=reranker_score))
        return results

    def touch_results(self, results: list[MemorySearchResult], policy: HumanMemoryPolicy) -> None:
        documents = []
        for result in results:
            record = policy.touch(result.record)
            documents.append({
                "id": record.id,
                "last_accessed_at": record.last_accessed_at,
                "access_count": record.access_count,
                "retention_score": record.retention_score,
                "status": str(record.status),
            })
        if documents:
            response = self.search_client.merge_documents(documents)
            self._raise_if_failed(response, "touch_results", "batch")

    def apply_forgetting(self, policy: HumanMemoryPolicy, *, limit: int = 100, dry_run: bool = True) -> list[dict[str, Any]]:
        results = self.search_client.search(
            search_text="*",
            filter="status ne 'deleted'",
            top=limit,
            order_by=["retention_score asc"],
        )
        updates = []
        for document in results:
            record = MemoryRecord.from_search_document(dict(document))
            new_score = policy.retention_score(record)
            new_status = policy.status_for(record)
            updates.append({"id": record.id, "memory": record.memory, "retention_score": new_score, "status": str(new_status)})
        if not dry_run and updates:
            response = self.search_client.merge_documents([
                {"id": item["id"], "retention_score": item["retention_score"], "status": item["status"]}
                for item in updates
            ])
            self._raise_if_failed(response, "apply_forgetting", "batch")
        return updates

    def _memory_scoring_profiles(self) -> list[ScoringProfile]:
        text_weights = TextWeights(weights={"memory": 3.0, "summary": 2.0, "category": 1.2})
        priority_functions = [
            MagnitudeScoringFunction(
                field_name="importance",
                boost=8.0,
                parameters=MagnitudeScoringParameters(
                    boosting_range_start=0.0,
                    boosting_range_end=1.0,
                    should_boost_beyond_range_by_constant=True,
                ),
                interpolation="linear",
            ),
            MagnitudeScoringFunction(
                field_name="retention_score",
                boost=4.0,
                parameters=MagnitudeScoringParameters(
                    boosting_range_start=0.0,
                    boosting_range_end=1.0,
                    should_boost_beyond_range_by_constant=True,
                ),
                interpolation="linear",
            ),
            MagnitudeScoringFunction(
                field_name="access_count",
                boost=4.0,
                parameters=MagnitudeScoringParameters(
                    boosting_range_start=0.0,
                    boosting_range_end=20.0,
                    should_boost_beyond_range_by_constant=True,
                ),
                interpolation="logarithmic",
            ),
        ]
        recency_functions = [
            FreshnessScoringFunction(
                field_name="updated_at",
                boost=8.0,
                parameters=FreshnessScoringParameters(boosting_duration=timedelta(days=90)),
                interpolation="linear",
            ),
        ]
        return [
            ScoringProfile(
                name=MEMORY_PRIORITY_SCORING_PROFILE,
                text_weights=text_weights,
                functions=priority_functions,
                function_aggregation="sum",
            ),
            ScoringProfile(
                name=MEMORY_RECENCY_SCORING_PROFILE,
                text_weights=text_weights,
                functions=recency_functions,
                function_aggregation="sum",
            ),
        ]

    def _ensure_scoring_profiles(self) -> None:
        index = self.index_client.get_index(self.index_name)
        desired_profiles = {profile.name: profile for profile in self._memory_scoring_profiles()}
        existing_profiles = [
            profile for profile in (index.scoring_profiles or [])
            if profile.name not in desired_profiles
        ]
        index.scoring_profiles = existing_profiles + list(desired_profiles.values())
        index.default_scoring_profile = None
        self.index_client.create_or_update_index(index)

    def _scoring_profile_for(self, options: MemorySearchOptions) -> str | None:
        if options.ranking_profile == RankingProfile.IMPORTANT:
            return MEMORY_PRIORITY_SCORING_PROFILE
        if options.ranking_profile == RankingProfile.RECENT:
            return MEMORY_RECENCY_SCORING_PROFILE
        return None

    def _order_by_for(self, options: MemorySearchOptions) -> str | None:
        return options.order_by

    def _filter_expression(self, *, user_id: str, agent_id: str, options: MemorySearchOptions) -> str:
        parts = [
            f"user_id eq '{self._odata_quote(user_id)}'",
            f"agent_id eq '{self._odata_quote(agent_id)}'",
        ]
        statuses = [MemoryStatus.ACTIVE]
        if options.include_archived:
            statuses.append(MemoryStatus.ARCHIVED)
        status_csv = ",".join(str(status) for status in statuses)
        parts.append(f"search.in(status, '{status_csv}', ',')")
        if options.category:
            parts.append(f"category eq '{self._odata_quote(options.category)}'")
        if options.min_importance is not None:
            parts.append(f"importance ge {options.min_importance}")
        if options.min_retention_score is not None:
            parts.append(f"retention_score ge {options.min_retention_score}")
        return " and ".join(parts)

    def _odata_quote(self, value: str) -> str:
        return value.replace("'", "''")

    def _raise_if_failed(self, response: list[Any], operation: str, target: str) -> None:
        failed = [item for item in response if not getattr(item, "succeeded", False)]
        if failed:
            details = ", ".join(str(getattr(item, "error_message", "unknown")) for item in failed)
            raise RuntimeError(f"{operation} failed for {target}: {details}")

print("AzureAISearchMemoryStore を定義しました。")

## 6. カスタム `ContextProvider`

この章では、ここまで作った部品を Agent Framework の実行サイクルへつなぎます。`ContextProvider` は、エージェントの実行前後に処理を差し込むための仕組みです。

実際にエージェントを呼び出すと、記憶まわりの処理は次の順番で動きます。

| タイミング | トリガー | このノートブックで行うこと | スコアや忘却への影響 |
|---|---|---|---|
| エージェント呼び出し前 | `before_run()` | ユーザー入力に関連する記憶を検索し、見つかった記憶をコンテキストへ追加する | `search_options.mode` と `search_options.ranking_profile` に従って Azure AI Search で検索し、検索で使われた記憶だけ `touch_results()` で想起済みにする |
| エージェント応答中 | モデル呼び出し | `before_run()` で追加された記憶を含めて回答を生成する | このタイミングでは記憶ストアを更新しない |
| エージェント応答後 | `after_run()` | 今回の入力と応答から記憶候補を抽出し、既存記憶との追加、更新、削除を判断する | 追加または更新された記憶は、その場で `retention_score` を再計算して保存する |
| 応答後の確認 | `apply_forgetting_after_run=True` | `apply_forgetting(..., dry_run=True)` を呼び、古くなった記憶の候補を確認する | このノートブックの既定ではプレビューだけで、Azure AI Search には忘却結果を保存しない |

ここで大事なのは、エージェント呼び出しのたびに全記憶を再計算しているわけではない、という点です。呼び出し前に更新されるのは検索で実際に使われた記憶だけです。呼び出し後に更新されるのは、新しく追加された記憶と、統合判断で更新された既存記憶だけです。

忘却処理も、既定では「実行後に候補を見る」だけです。`dry_run=True` のため、`retention_score` や `status` の見込み値は計算しますが、検索インデックスへは書き戻しません。実運用で本当に古い記憶を `archived` に進める場合は、`apply_forgetting(..., dry_run=False)` を定期ジョブや低頻度のメンテナンス処理として実行する設計にします。

`search_options` は `before_run()` の検索方針として使われます。既定では `SearchMode.HYBRID`、`RankingProfile.IMPORTANT`、`top=5`、`min_retention_score=0.1` を使い、関連しそうで、かつ保持スコアが低すぎない記憶だけをエージェントに渡します。最近更新された条件を検索順位へ強く反映したい場合は、`mode` はそのままに `ranking_profile=RankingProfile.RECENT` を選びます。

In [ ]:
# pyright: reportMissingImports=false, reportUnusedImport=false

try:
    from agent_framework import Message
    from agent_framework._sessions import ContextProvider
except Exception:
    @dataclass
    class Message:
        role: str
        contents: list[str]
        message_id: str | None = None
        author_name: str | None = None
        additional_properties: dict[str, Any] = field(default_factory=dict)

        @property
        def text(self) -> str:
            return "\n".join(str(item) for item in self.contents)

    class ContextProvider:
        def __init__(self, source_id: str):
            self.source_id = source_id


class AzureAISearchMemoryContextProvider(ContextProvider):
    DEFAULT_CONTEXT_PROMPT = "## Relevant memories\nUse these memories only when they help answer the user."

    def __init__(
        self,
        *,
        store: AzureAISearchMemoryStore,
        intelligence: AzureOpenAIMemoryIntelligence,
        extractor: MemoryExtractor,
        consolidator: MemoryConsolidator,
        policy: HumanMemoryPolicy,
        user_id: str,
        agent_id: str,
        source_id: str = "custom_aisearch_memory",
        search_options: MemorySearchOptions | None = None,
        context_prompt: str | None = None,
        apply_forgetting_after_run: bool = True,
        log_search_results: bool = True,
        log_empty_search_diagnostics: bool = True,
    ):
        super().__init__(source_id)
        self.store = store
        self.intelligence = intelligence
        self.extractor = extractor
        self.consolidator = consolidator
        self.policy = policy
        self.user_id = user_id
        self.agent_id = agent_id
        self.search_options = search_options or MemorySearchOptions(mode=SearchMode.HYBRID, top=5, min_retention_score=0.1)
        self.context_prompt = context_prompt or self.DEFAULT_CONTEXT_PROMPT
        self.apply_forgetting_after_run = apply_forgetting_after_run
        self.log_search_results = log_search_results
        self.log_empty_search_diagnostics = log_empty_search_diagnostics

    async def before_run(
        self,
        *,
        agent: Any,
        session: Any,
        context: Any,
        state: dict[str, Any],
    ) -> None:
        _ = (agent, session)
        query = "\n".join(message.text for message in context.input_messages if getattr(message, "text", "").strip())
        if not query.strip():
            return
        query_vector = self.intelligence.embed(query)
        results = self.store.search(
            query=query,
            query_vector=query_vector,
            user_id=self.user_id,
            agent_id=self.agent_id,
            options=self.search_options,
        )
        self.store.touch_results(results, self.policy)
        state["last_memory_search"] = [self._search_result_to_state(result) for result in results]
        self._log_automatic_search(query, results)
        if results:
            context.extend_messages(self, [Message(role="user", contents=[self._format_context(results)])])

    async def after_run(
        self,
        *,
        agent: Any,
        session: Any,
        context: Any,
        state: dict[str, Any],
    ) -> None:
        _ = (agent, session)
        messages = [self._message_to_dict(message) for message in context.input_messages]
        if context.response and context.response.messages:
            messages.extend(self._message_to_dict(message) for message in context.response.messages)
        decisions = await self.ingest_messages(messages, session_id=context.session_id)
        state["last_memory_decisions"] = [self._decision_to_state(decision) for decision in decisions]
        if self.apply_forgetting_after_run:
            state["forgetting_preview"] = self.store.apply_forgetting(self.policy, dry_run=True, limit=25)

    async def ingest_messages(self, messages: list[dict[str, str]], *, session_id: str | None = None) -> list[ConsolidationDecision]:
        candidates = self.extractor.extract(messages)
        decisions: list[ConsolidationDecision] = []
        for candidate in candidates:
            candidate_vector = self.intelligence.embed(candidate.memory)
            neighbors = self.store.search(
                query=candidate.memory,
                query_vector=candidate_vector,
                user_id=self.user_id,
                agent_id=self.agent_id,
                options=MemorySearchOptions(mode=SearchMode.HYBRID, top=5, include_archived=True),
            )
            decision = self.consolidator.decide(candidate, neighbors)
            self._apply_decision(decision, candidate_vector, session_id=session_id)
            decisions.append(decision)
        return decisions

    def search_memories(self, query: str, options: MemorySearchOptions | None = None) -> list[MemorySearchResult]:
        options = options or self.search_options
        return self.store.search(
            query=query,
            query_vector=self.intelligence.embed(query),
            user_id=self.user_id,
            agent_id=self.agent_id,
            options=options,
        )

    def _apply_decision(
        self,
        decision: ConsolidationDecision,
        candidate_vector: list[float],
        *,
        session_id: str | None,
    ) -> None:
        now = utc_now()
        if decision.event == MemoryEvent.NONE:
            return
        if decision.event == MemoryEvent.DELETE and decision.target_id:
            self.store.mark_status(decision.target_id, MemoryStatus.DELETED)
            return
        if decision.event == MemoryEvent.UPDATE and decision.target_id:
            existing = self.store.get(decision.target_id)
            if not existing:
                return
            existing.memory = decision.merged_memory or decision.candidate.memory
            existing.summary = existing.memory
            existing.category = decision.candidate.category or existing.category
            existing.importance = max(existing.importance, decision.candidate.importance)
            existing.confidence = max(existing.confidence, decision.candidate.confidence)
            existing.updated_at = now
            existing.expires_at = decision.candidate.expires_at or existing.expires_at
            existing.metadata.update(decision.candidate.metadata)
            existing.metadata["last_consolidation_reason"] = decision.reason
            existing.retention_score = self.policy.retention_score(existing, now=now)
            self.store.upsert(existing, embedding=self.intelligence.embed(existing.memory))
            return
        if decision.event == MemoryEvent.ADD:
            record = MemoryRecord(
                id=str(uuid.uuid4()),
                user_id=self.user_id,
                agent_id=self.agent_id,
                memory=decision.candidate.memory,
                summary=decision.candidate.memory,
                category=decision.candidate.category,
                importance=decision.candidate.importance,
                confidence=decision.candidate.confidence,
                source=decision.candidate.source,
                created_at=now,
                updated_at=now,
                expires_at=decision.candidate.expires_at,
                metadata={**decision.candidate.metadata, "session_id": session_id, "consolidation_reason": decision.reason},
            )
            record.retention_score = self.policy.retention_score(record, now=now)
            self.store.upsert(record, embedding=candidate_vector)

    def _format_context(self, results: list[MemorySearchResult]) -> str:
        lines = [self.context_prompt]
        for index, result in enumerate(results, start=1):
            record = result.record
            lines.append(
                f"{index}. [{record.category}; importance={record.importance:.2f}; retention={record.retention_score:.2f}] "
                f"{record.memory}"
            )
        return "\n".join(lines)

    def _log_automatic_search(self, query: str, results: list[MemorySearchResult]) -> None:
        if not self.log_search_results:
            return
        filter_expression = self.store._filter_expression(
            user_id=self.user_id,
            agent_id=self.agent_id,
            options=self.search_options,
        )
        payload = {
            "source_id": self.source_id,
            "user_id": self.user_id,
            "agent_id": self.agent_id,
            "query": query,
            "mode": str(self.search_options.mode),
            "top": self.search_options.top,
            "filter": filter_expression,
            "min_retention_score": self.search_options.min_retention_score,
            "include_archived": self.search_options.include_archived,
            "result_count": len(results),
            "results": [self._search_result_to_log_item(index, result) for index, result in enumerate(results, start=1)],
        }
        if not results and self.log_empty_search_diagnostics:
            payload["empty_result_diagnostics"] = self._empty_search_diagnostics(query)
        custom_memory_logger.info("Automatic memory search results:\n%s", json.dumps(payload, ensure_ascii=False, indent=2))

    def _search_result_to_log_item(self, index: int, result: MemorySearchResult) -> dict[str, Any]:
        record = result.record
        return {
            "rank": index,
            "id": record.id,
            "score": result.score,
            "reranker_score": result.reranker_score,
            "memory": record.memory,
            "category": record.category,
            "status": str(record.status),
            "importance": record.importance,
            "retention_score": record.retention_score,
            "access_count": record.access_count,
            "updated_at": dt_to_json(record.updated_at),
            "last_accessed_at": dt_to_json(record.last_accessed_at),
        }

    def _empty_search_diagnostics(self, query: str) -> dict[str, Any]:
        diagnostics: dict[str, Any] = {}
        try:
            relaxed_options = MemorySearchOptions(
                mode=SearchMode.KEYWORD,
                top=5,
                category=self.search_options.category,
                include_archived=True,
            )
            relaxed_results = self.store.search(
                query=query,
                query_vector=None,
                user_id=self.user_id,
                agent_id=self.agent_id,
                options=relaxed_options,
            )
            any_memory_results = self.store.search(
                query="*",
                query_vector=None,
                user_id=self.user_id,
                agent_id=self.agent_id,
                options=relaxed_options,
            )
            diagnostics["relaxed_filter"] = self.store._filter_expression(
                user_id=self.user_id,
                agent_id=self.agent_id,
                options=relaxed_options,
            )
            diagnostics["same_query_without_retention_filter_count"] = len(relaxed_results)
            diagnostics["same_query_without_retention_filter_results"] = [
                self._search_result_to_log_item(index, result)
                for index, result in enumerate(relaxed_results, start=1)
            ]
            diagnostics["visible_memory_sample_count"] = len(any_memory_results)
            diagnostics["visible_memory_sample"] = [
                self._search_result_to_log_item(index, result)
                for index, result in enumerate(any_memory_results, start=1)
            ]
        except Exception as error:
            diagnostics["error"] = repr(error)
        return diagnostics

    def _message_to_dict(self, message: Any) -> dict[str, str]:
        role = str(getattr(message, "role", "user"))
        text = str(getattr(message, "text", "") or "")
        return {"role": role, "content": text}

    def _search_result_to_state(self, result: MemorySearchResult) -> dict[str, Any]:
        return {
            "id": result.record.id,
            "memory": result.record.memory,
            "category": result.record.category,
            "score": result.score,
            "importance": result.record.importance,
            "retention_score": result.record.retention_score,
        }

    def _decision_to_state(self, decision: ConsolidationDecision) -> dict[str, Any]:
        return {
            "event": str(decision.event),
            "target_id": decision.target_id,
            "memory": decision.merged_memory or decision.candidate.memory,
            "reason": decision.reason,
        }

print("AzureAISearchMemoryContextProvider を定義しました。")

## 7. メモリー プロバイダーを初期化する

このセルでは、記憶抽出、統合判断、忘却ポリシー、Azure AI Search ストアを組み合わせて `custom_memory_provider` を作ります。以降のセルはこのプロバイダーを使って、記憶の追加、検索、忘却の挙動を確認します。

初期化時に渡している `MemorySearchOptions(mode=SearchMode.HYBRID, top=5, min_retention_score=0.1)` は、エージェント実行前の既定検索です。ハイブリッド検索で関連記憶を探し、最大 5 件だけをコンテキストに入れ、保持スコアが低すぎる記憶は除外します。

`CUSTOM_MEMORY_RESET_INDEX=true` を設定すると、検証用インデックスを削除して作り直します。既存インデックスにもスコアリング プロファイルは追補しますが、スキーマ全体を変えた直後は、再作成すると挙動を確認しやすくなります。

In [ ]:
# pyright: reportMissingImports=false

memory_intelligence = AzureOpenAIMemoryIntelligence(
    endpoint=AZURE_OPENAI_ENDPOINT,
    api_key=AZURE_OPENAI_API_KEY,
    api_version=AZURE_OPENAI_API_VERSION,
    chat_deployment=AZURE_OPENAI_CHAT_DEPLOYMENT,
    embedding_deployment=AZURE_OPENAI_EMBEDDING_DEPLOYMENT,
    embedding_dims=AZURE_OPENAI_EMBEDDING_DIMS,
)
memory_extractor = MemoryExtractor(memory_intelligence)
memory_policy = HumanMemoryPolicy(half_life_days=45, access_boost=0.04)
memory_consolidator = MemoryConsolidator(memory_intelligence)

memory_store = None
custom_memory_provider = None

if not SEARCH_READY:
    print("スキップ: Azure AI Search の endpoint または index 名が不足しています。")
else:
    if os.getenv("CUSTOM_MEMORY_RESET_INDEX", "false").casefold() in {"1", "true", "yes", "y"}:
        credential = AzureKeyCredential(AZURE_SEARCH_API_KEY) if AZURE_SEARCH_API_KEY else DefaultAzureCredential()
        reset_client = SearchIndexClient(endpoint=AZURE_SEARCH_ENDPOINT, credential=credential)
        if AZURE_SEARCH_INDEX_NAME in set(reset_client.list_index_names()):
            reset_client.delete_index(AZURE_SEARCH_INDEX_NAME)
            print("既存 index を削除しました:", AZURE_SEARCH_INDEX_NAME)
        reset_client.close()

    memory_store = AzureAISearchMemoryStore(
        endpoint=AZURE_SEARCH_ENDPOINT,
        index_name=AZURE_SEARCH_INDEX_NAME,
        api_key=AZURE_SEARCH_API_KEY,
        embedding_dims=AZURE_OPENAI_EMBEDDING_DIMS,
    )
    custom_memory_provider = AzureAISearchMemoryContextProvider(
        store=memory_store,
        intelligence=memory_intelligence,
        extractor=memory_extractor,
        consolidator=memory_consolidator,
        policy=memory_policy,
        user_id=DEMO_USER_ID,
        agent_id=DEMO_AGENT_ID,
        search_options=MemorySearchOptions(mode=SearchMode.HYBRID, top=5, min_retention_score=0.1),
    )
    print("Custom memory provider を初期化しました。")
    print("  index:", AZURE_SEARCH_INDEX_NAME)
    print("  user_id:", DEMO_USER_ID)
    print("  agent_id:", DEMO_AGENT_ID)
    print("  chat_llm:", memory_intelligence.can_chat)
    print("  embedding:", memory_intelligence.can_embed or "stable_hash_fallback")

## 8. 記憶を追加し、複数検索方式で見る

このセルでは `ingest_messages()` を直接呼び、会話から記憶候補を抽出して Azure AI Search に保存します。実アプリでは `after_run()` が同じ流れを呼びます。

後半では、同じ質問に対して `SearchMode` と `RankingProfile` の組み合わせを切り替え、検索結果の違いを見ます。ここで見てほしいのは「どの検索が正しいか」ではなく、「候補取得の方式」と「順位付けの方針」は別の軸だという点です。呼び出し側は Azure AI Search のスコアリング プロファイル名ではなく、`RankingProfile` を指定します。

| 検索方式 | 何を重視するか | 向いている場面 |
|---|---|---|
| `SearchMode.KEYWORD` | 入力語句と記憶本文の一致 | 固有名詞、商品名、明確なキーワードを探す |
| `SearchMode.VECTOR` | 意味の近さ | 言い換えや曖昧な質問から関連記憶を探す |
| `SearchMode.HYBRID` | キーワード検索とベクトル検索の組み合わせ | 通常のエージェント記憶検索の既定候補 |
| `SearchMode.SEMANTIC` | ハイブリッド検索にセマンティック ランカーを加える | 検索結果の説明文や順位の品質を高めたい時 |

| 順位付け方針 | 内部で使う Azure AI Search profile | 向いている場面 |
|---|---|---|
| `RankingProfile.NONE` | なし | 純粋な keyword / vector / hybrid の差を見たい時 |
| `RankingProfile.IMPORTANT` | `memory-priority` | 長期的な好みや強い制約を優先したい時 |
| `RankingProfile.RECENT` | `memory-recency` | 直近の予定や最近追加された条件を優先したい時 |

このセルでは、まず `RankingProfile.NONE` で検索方式ごとの差を見ます。その後、同じ `SearchMode.HYBRID` に `RankingProfile.IMPORTANT` と `RankingProfile.RECENT` を組み合わせ、スコアリング プロファイルによる順位差を確認します。

In [ ]:
demo_messages = [
    {"role": "user", "content": "私の名前は Haruka です。大阪への週末旅行では、駅近ホテルと通路側の席を優先したいです。"},
    {"role": "assistant", "content": "大阪旅行では駅近ホテルと通路側席を優先して提案します。"},
    {"role": "user", "content": "食事はベジタリアン対応がある店を選びたいです。古い予定は来月以降は忘れてよいです。"},
]

if custom_memory_provider is None:
    print("スキップ: custom_memory_provider が初期化されていません。")
else:
    decisions = await custom_memory_provider.ingest_messages(demo_messages, session_id="notebook-demo")
    print("統合判断:")
    print(json.dumps([custom_memory_provider._decision_to_state(decision) for decision in decisions], ensure_ascii=False, indent=2))

    query = "大阪旅行でホテルと食事を選ぶ時の好み"
    comparisons = [
        ("KEYWORD", MemorySearchOptions(mode=SearchMode.KEYWORD, ranking_profile=RankingProfile.NONE, top=5, include_archived=True)),
        ("VECTOR", MemorySearchOptions(mode=SearchMode.VECTOR, ranking_profile=RankingProfile.NONE, top=5, include_archived=True)),
        ("HYBRID", MemorySearchOptions(mode=SearchMode.HYBRID, ranking_profile=RankingProfile.NONE, top=5, include_archived=True)),
        ("SEMANTIC", MemorySearchOptions(mode=SearchMode.SEMANTIC, ranking_profile=RankingProfile.NONE, top=5, include_archived=True)),
        ("HYBRID + IMPORTANT", MemorySearchOptions(mode=SearchMode.HYBRID, ranking_profile=RankingProfile.IMPORTANT, top=5, include_archived=True)),
        ("HYBRID + RECENT", MemorySearchOptions(mode=SearchMode.HYBRID, ranking_profile=RankingProfile.RECENT, top=5, include_archived=True)),
    ]
    for label, options in comparisons:
        results = custom_memory_provider.search_memories(query, options=options)
        print(f"\n--- {label} ---")
        print(json.dumps([
            {
                "score": result.score,
                "reranker_score": result.reranker_score,
                "mode": str(options.mode),
                "ranking_profile": str(options.ranking_profile),
                "memory": result.record.memory,
                "category": result.record.category,
                "importance": result.record.importance,
                "retention_score": result.record.retention_score,
                "created_at": dt_to_json(result.record.created_at),
                "updated_at": dt_to_json(result.record.updated_at),
            }
            for result in results
        ], ensure_ascii=False, indent=2))

## 9. 人間らしい記憶: 想起、減衰、忘却

この章では、保存した記憶をずっと同じ重さで扱わないための制御を確認します。エージェントの記憶は、増えるほど便利になる一方で、古い予定や弱い好みまで毎回参照すると回答品質を下げます。そのため、思い出された記憶は残しやすくし、使われない記憶は徐々に見えにくくします。

ここからのシナリオでは、状態更新だけでなく `RankingProfile` によって Azure AI Search の順位付け方針がどう変わるかも確認します。`RankingProfile.IMPORTANT` は重要度、保持スコア、参照回数をブーストし、`RankingProfile.RECENT` は最近更新された記憶をより強く上位に寄せます。検索方式は引き続き `SearchMode` で指定します。

| 機能 | 実装 | 検索結果への影響 |
|---|---|---|
| 想起による強化 | 検索で使われた記憶の `access_count` と `last_accessed_at` を更新 | `RankingProfile.IMPORTANT` で、よく思い出される記憶が上位に寄りやすくなる |
| IMPORTANT と RECENT の比較 | 古いが重要な記憶と、新しいが軽い記憶を同じクエリで比べる | 同じ検索方式でも、ranking profile によって上位結果が変わる |
| 時間による減衰 | `updated_at` からの経過日数と半減期で `retention_score` を下げる | 古く弱い記憶は順位が下がり、状態も `archived` 側へ遷移する |
| 古い記憶の扱い | `active` から `archived` へ状態遷移させる | フィルターで既定検索から外し、必要な確認時だけ検索対象に戻す |
| 明示的な削除 | ユーザーの忘却指示や矛盾時は `deleted` にする | スコアが高くてもフィルターで検索結果から外す |

下のシナリオコードは、専用テストデータを作って次を順に確認します。

1. 記憶を作成し、スコアリング確認用の要約関数を用意する
2. `RankingProfile.IMPORTANT` で検索し、想起前後の順位とスコアを見る
3. `RankingProfile.IMPORTANT` と `RankingProfile.RECENT` で、古いが重要な記憶と新しい一時候補の順位差を見る
4. 1 件を `deleted` にして、スコアではなくフィルターで除外されることを確認する
5. 忘却候補と、ユーザー応答向けの検索結果の違いを確認する

### 9-0b. シナリオ用データを初期化する

このセルは、9章の各確認セルが参照するテスト用の記憶を12件作ります。あえて「大阪旅行」「ホテル」「駅近」「荷物」「イベント」「計画」などの近い語を含む記憶を混ぜ、検索結果の順位差が見えるようにしています。

`scenario_ctx` には後続セルで使うカテゴリ名と各 document ID を入れます。検索結果には `role` も出すので、どの記憶がターゲットで、どれが似ているだけの候補かを確認できます。

`IMPORTANT` と `RECENT` の違いを見るため、本文の検索語をそろえた対照ペアも入れています。

- `IMPORTANT_TARGET`: 同じ検索語を持つ、古いが重要度と参照回数が高い基本方針
- `RECENT_TARGET`: 同じ検索語を持つ、新しいが重要度の低い一時候補

In [ ]:
def delete_all_memory_documents(store: AzureAISearchMemoryStore, *, batch_size: int = 1000) -> int:
    document_ids: list[str] = []
    seen_ids: set[str] = set()
    for document in store.search_client.search(search_text="*", select=["id"]):
        document_id = str(document["id"])
        if document_id not in seen_ids:
            seen_ids.add(document_id)
            document_ids.append(document_id)

    for start in range(0, len(document_ids), batch_size):
        batch_ids = document_ids[start:start + batch_size]
        response = store.search_client.delete_documents(documents=[{"id": document_id} for document_id in batch_ids])
        store._raise_if_failed(response, "delete_documents", "batch")

    return len(document_ids)


scenario_ctx = {}

if memory_store is None:
    print("スキップ: memory_store が初期化されていません。")
else:
    deleted_count = delete_all_memory_documents(memory_store)
    print(f"対象 index の document を削除しました: {deleted_count} 件")
    print("シナリオ用の状態もクリアしました。")

In [ ]:
import time


def summarize_scored_results(results: list[MemorySearchResult]) -> list[dict[str, Any]]:
    return [
        {
            "rank": index,
            "id": result.record.id,
            "role": result.record.metadata.get("scenario_role"),
            "score": result.score,
            "reranker_score": result.reranker_score,
            "memory": result.record.memory,
            "importance": result.record.importance,
            "retention_score": result.record.retention_score,
            "access_count": result.record.access_count,
            "status": str(result.record.status),
            "updated_at": dt_to_json(result.record.updated_at),
            "last_accessed_at": dt_to_json(result.record.last_accessed_at),
        }
        for index, result in enumerate(results, start=1)
    ]


def wait_for_scenario_documents(store: AzureAISearchMemoryStore, *, category: str, expected_count: int, timeout_seconds: float = 15.0) -> int:
    deadline = time.monotonic() + timeout_seconds
    observed_count = 0
    while time.monotonic() < deadline:
        documents = list(
            store.search_client.search(
                search_text="*",
                filter=f"category eq '{store._odata_quote(category)}'",
                select=["id"],
                top=expected_count,
            )
        )
        observed_count = len(documents)
        if observed_count >= expected_count:
            return observed_count
        time.sleep(1.0)
    return observed_count


scenario_ctx = {}

if memory_store is None:
    print("スキップ: memory_store が初期化されていません。")
else:
    scenario_tag = f"memory-lifecycle-{uuid.uuid4().hex[:8]}"
    now = utc_now()
    seed_specs = [
        {
            "name": "record_recall_id",
            "role": "TARGET: queryに完全一致する駅近ホテル条件",
            "memory": "大阪旅行では駅近ホテルを最優先し、チェックイン前に荷物預かりできるホテルを選ぶ。",
            "importance": 0.95,
            "days_old": 7,
            "access_count": 0,
        },
        {
            "name": "record_hotel_budget_id",
            "role": "SIMILAR: ホテルだが料金条件",
            "memory": "大阪旅行ではホテル料金は1泊2万円以内を目安にする。",
            "importance": 0.65,
            "days_old": 4,
            "access_count": 0,
        },
        {
            "name": "record_hotel_view_id",
            "role": "SIMILAR: ホテルだが眺望条件",
            "memory": "大阪旅行では夜景が見える高層ホテルも候補に入れる。",
            "importance": 0.55,
            "days_old": 3,
            "access_count": 0,
        },
        {
            "name": "record_station_locker_id",
            "role": "SIMILAR: 駅近/荷物だがホテルではない",
            "memory": "大阪旅行では駅近のコインロッカーに荷物を預ける選択肢も確認する。",
            "importance": 0.50,
            "days_old": 2,
            "access_count": 0,
        },
        {
            "name": "record_recent_id",
            "role": "RECENT: 最新イベント確認用",
            "memory": "大阪のイベント情報は直前に公式サイトで確認する。",
            "importance": 0.45,
            "days_old": 1,
            "access_count": 0,
        },
        {
            "name": "record_important_policy_id",
            "role": "IMPORTANT_TARGET: 同じ本文で古いが重要で何度も参照された基本方針",
            "memory": "大阪旅行のホテルの基本方針はたこ焼き屋が徒歩圏内であること。",
            "importance": 1.0,
            "days_old": 75,
            "access_count": 2,
        },
        {
            "name": "record_recent_note_id",
            "role": "RECENT_TARGET: 同じ本文で新しいが重要度の低い一時候補",
            "memory": "大阪旅行のホテルの基本方針はたこ焼き屋が徒歩圏内であること。",
            "importance": 0.05,
            "days_old": 0,
            "access_count": 2,
        },
        {
            "name": "record_food_id",
            "role": "DISTRACTOR: 食事条件",
            "memory": "大阪旅行ではベジタリアン対応の店を優先して探す。",
            "importance": 0.70,
            "days_old": 5,
            "access_count": 0,
        },
        {
            "name": "record_seat_id",
            "role": "DISTRACTOR: 移動中の座席条件",
            "memory": "大阪旅行の新幹線では通路側の席を優先する。",
            "importance": 0.60,
            "days_old": 6,
            "access_count": 0,
        },
        {
            "name": "record_transport_id",
            "role": "DISTRACTOR: 市内移動条件",
            "memory": "大阪市内の移動では地下鉄を優先し、タクシーは雨の日だけ使う。",
            "importance": 0.50,
            "days_old": 8,
            "access_count": 0,
        },
        {
            "name": "record_forgetting_id",
            "role": "OLD: 古いイベント候補",
            "memory": "大阪旅行では半年前に見つけたイベント候補も念のため確認する。",
            "importance": 0.20,
            "days_old": 140,
            "access_count": 0,
        },
        {
            "name": "record_old_hotel_id",
            "role": "OLD: 古いホテル候補",
            "memory": "大阪旅行では昔泊まった郊外のホテルも候補に入れる。",
            "importance": 0.25,
            "days_old": 180,
            "access_count": 0,
        },
    ]

    seed_records: dict[str, MemoryRecord] = {}
    for spec in seed_specs:
        record = MemoryRecord(
            id=str(uuid.uuid4()),
            user_id=DEMO_USER_ID,
            agent_id=DEMO_AGENT_ID,
            memory=spec["memory"],
            summary=spec["memory"],
            category=scenario_tag,
            importance=float(spec["importance"]),
            confidence=0.90,
            created_at=now - timedelta(days=int(spec["days_old"])),
            updated_at=now - timedelta(days=int(spec["days_old"])),
            access_count=int(spec["access_count"]),
            metadata={"scenario_role": spec["role"]},
        )
        record.retention_score = memory_policy.retention_score(record, now=now)
        record.status = memory_policy.status_for(record, now=now)
        seed_records[spec["name"]] = record
        memory_store.upsert(record, embedding=memory_intelligence.embed(record.memory))

    observed_count = wait_for_scenario_documents(memory_store, category=scenario_tag, expected_count=len(seed_records))
    scenario_ctx = {
        "scenario_tag": scenario_tag,
        **{name: record.id for name, record in seed_records.items()},
    }

    print("シナリオ用 document を作成しました。")
    print(json.dumps(
        {
            "scenario_tag": scenario_ctx["scenario_tag"],
            "expected_count": len(seed_records),
            "search_visible_count": observed_count,
            "target_memory": seed_records["record_recall_id"].memory,
            "important_target": seed_records["record_important_policy_id"].memory,
            "recent_target": seed_records["record_recent_note_id"].memory,
            "record_ids": {name: record.id for name, record in seed_records.items()},
        },
        ensure_ascii=False,
        indent=2,
    ))

### 9-1. 想起による強化をスコアで確認する

このセルでは、`大阪旅行 駅近ホテル 荷物預かり チェックイン前` という具体的なクエリを `SearchMode.HYBRID` と `RankingProfile.IMPORTANT` で検索します。シナリオ データには似た記憶を混ぜているため、検索結果の `role` を見ると、どの記憶が本命で、どれが似ているだけかを確認できます。

確認ポイント:

- `RankingProfile.IMPORTANT` が内部で `memory-priority` を使う
- `TARGET: queryに完全一致する駅近ホテル条件` を明示的に想起して touch する
- `touched_expected_target` が `true` になる
- Azure AI Search へ更新が反映された後、touch された記憶だけ `access_count`、`retention_score`、`last_accessed_at` が変わる

In [ ]:
if "scenario_ctx" not in globals() or not scenario_ctx:
    print("スキップ: シナリオ初期化セルを先に実行してください。")
else:
    query = "大阪旅行 駅近ホテル 荷物預かり チェックイン前"
    query_vector = memory_intelligence.embed(query)
    priority_options = MemorySearchOptions(
        mode=SearchMode.HYBRID,
        ranking_profile=RankingProfile.IMPORTANT,
        top=5,
        category=scenario_ctx["scenario_tag"],
        include_archived=True,
    )
    expected_id = scenario_ctx["record_recall_id"]
    expected_before_touch = memory_store.get(expected_id)
    previous_access_count = expected_before_touch.access_count if expected_before_touch else -1

    results_before_touch = memory_store.search(
        query=query,
        query_vector=query_vector,
        user_id=DEMO_USER_ID,
        agent_id=DEMO_AGENT_ID,
        options=priority_options,
    )
    before_summary = summarize_scored_results(results_before_touch)

    recalled_results = [result for result in results_before_touch if result.record.id == expected_id]
    if not recalled_results and expected_before_touch:
        recalled_results = [MemorySearchResult(record=expected_before_touch)]

    memory_store.touch_results(recalled_results, memory_policy)

    expected_after_touch = None
    deadline = time.monotonic() + 5.0
    while time.monotonic() < deadline:
        candidate = memory_store.get(expected_id)
        if candidate and candidate.access_count > previous_access_count:
            expected_after_touch = candidate
            break
        time.sleep(0.5)
    if expected_after_touch is None:
        expected_after_touch = memory_store.get(expected_id)

    touched_ids = [result.record.id for result in recalled_results]
    touched_expected_target = expected_id in touched_ids
    results_after_touch = memory_store.search(
        query=query,
        query_vector=query_vector,
        user_id=DEMO_USER_ID,
        agent_id=DEMO_AGENT_ID,
        options=priority_options,
    )

    print("[1] 想起による強化と important ranking")
    print(json.dumps(
        {
            "mode": str(priority_options.mode),
            "ranking_profile": str(priority_options.ranking_profile),
            "query": query,
            "expected_top_role": "TARGET: queryに完全一致する駅近ホテル条件",
            "expected_touched_id": expected_id,
            "touched_ids": touched_ids,
            "touched_expected_target": touched_expected_target,
            "expected_before_touch": summarize_scored_results([MemorySearchResult(record=expected_before_touch)]) if expected_before_touch else [],
            "expected_after_touch": summarize_scored_results([MemorySearchResult(record=expected_after_touch)]) if expected_after_touch else [],
            "before": before_summary,
            "after": summarize_scored_results(results_after_touch),
        },
        ensure_ascii=False,
        indent=2,
    ))

### 9-2. IMPORTANT と RECENT の順位差を確認する

このセルでは、同じ `SearchMode.HYBRID` に `RankingProfile.IMPORTANT` と `RankingProfile.RECENT` を組み合わせ、本文の検索語をそろえた2件を比較します。本文一致度をそろえることで、順位差が `importance`、`access_count`、`updated_at` の違いとして見えやすくなります。

- `IMPORTANT_TARGET`: 同じ検索語を持つ、古いが重要度と参照回数が高い基本方針
- `RECENT_TARGET`: 同じ検索語を持つ、新しいが重要度の低い一時候補

確認ポイント:

- `RankingProfile.IMPORTANT` では `IMPORTANT_TARGET` が上位に来やすい
- `RankingProfile.RECENT` では `RECENT_TARGET` が上位に来やすい
- 同じ検索方式でも、ranking profile によって順位付けの意図が変わる

In [ ]:
if "scenario_ctx" not in globals() or not scenario_ctx:
    print("スキップ: シナリオ初期化セルを先に実行してください。")
else:
    query = "大阪旅行 計画 方針"
    query_vector = memory_intelligence.embed(query)
    important_options = MemorySearchOptions(
        mode=SearchMode.HYBRID,
        ranking_profile=RankingProfile.IMPORTANT,
        top=10,
        category=scenario_ctx["scenario_tag"],
        include_archived=True,
    )
    recency_options = MemorySearchOptions(
        mode=SearchMode.HYBRID,
        ranking_profile=RankingProfile.RECENT,
        top=10,
        category=scenario_ctx["scenario_tag"],
        include_archived=True,
    )
    important_results = memory_store.search(
        query=query,
        query_vector=query_vector,
        user_id=DEMO_USER_ID,
        agent_id=DEMO_AGENT_ID,
        options=important_options,
    )
    recency_results = memory_store.search(
        query=query,
        query_vector=query_vector,
        user_id=DEMO_USER_ID,
        agent_id=DEMO_AGENT_ID,
        options=recency_options,
    )

    print("[2] IMPORTANT と RECENT の ranking profile 比較")
    print(json.dumps(
        {
            "query": query,
            "important_expected_role": "IMPORTANT_TARGET: 同じ検索語を持つ、古いが重要で何度も参照された基本方針",
            "recent_expected_role": "RECENT_TARGET: 同じ検索語を持つ、新しいが重要度の低い一時候補",
            "important_expected_id": scenario_ctx["record_important_policy_id"],
            "recent_expected_id": scenario_ctx["record_recent_note_id"],
            "mode": str(important_options.mode),
            "important_ranking_profile": str(important_options.ranking_profile),
            "important_results": summarize_scored_results(important_results),
            "recency_ranking_profile": str(recency_options.ranking_profile),
            "recency_results": summarize_scored_results(recency_results),
        },
        ensure_ascii=False,
        indent=2,
    ))

### 9-3. 明示的な削除を検索結果で確認する

このセルでは、対象記憶を `deleted` に変更し、その後に `SearchMode.HYBRID` と `RankingProfile.IMPORTANT` で検索します。

確認ポイント:

- 対象記憶の `status` が `deleted` になる
- 重要度が高くても、`deleted` はフィルターで検索結果から外れる

In [ ]:
if "scenario_ctx" not in globals() or not scenario_ctx:
    print("スキップ: シナリオ初期化セルを先に実行してください。")
else:
    memory_store.mark_status(scenario_ctx["record_recall_id"], MemoryStatus.DELETED)
    deleted_after = memory_store.get(scenario_ctx["record_recall_id"])
    visible_after_delete = memory_store.search(
        query="大阪旅行 ホテル 優先",
        query_vector=None,
        user_id=DEMO_USER_ID,
        agent_id=DEMO_AGENT_ID,
        options=MemorySearchOptions(
            mode=SearchMode.KEYWORD,
            ranking_profile=RankingProfile.IMPORTANT,
            top=10,
            category=scenario_ctx["scenario_tag"],
            include_archived=True,
        ),
    )

    print("[3] 明示的な削除と important ranking")
    print(json.dumps(
        {
            "deleted": {
                "id": scenario_ctx["record_recall_id"],
                "status": str(deleted_after.status) if deleted_after else None,
            },
            "mode": str(SearchMode.KEYWORD),
            "ranking_profile": str(RankingProfile.IMPORTANT),
            "visible_after_delete": summarize_scored_results(visible_after_delete),
        },
        ensure_ascii=False,
        indent=2,
    ))

### 9-4. 忘却 dry-run と応答向け検索を比較する

このセルでは `apply_forgetting(..., dry_run=True)` を使って、実際の更新をせず候補だけを確認します。

忘却候補の確認は、応答向けの順位付けとは目的が違います。忘却では弱い記憶を見つけたいので `retention_score asc` を使い、ユーザー応答では `RankingProfile.IMPORTANT` や `RankingProfile.RECENT` を使って有用な記憶を上位に寄せます。

確認ポイント:

- シナリオ対象の記憶が忘却候補としてどう評価されるか
- 忘却候補と、`RankingProfile.IMPORTANT` の上位結果が異なること

In [ ]:
if "scenario_ctx" not in globals() or not scenario_ctx:
    print("スキップ: シナリオ初期化セルを先に実行してください。")
else:
    forgetting_preview = memory_store.apply_forgetting(memory_policy, dry_run=True, limit=100)
    scenario_ids = {
        scenario_ctx["record_recall_id"],
        scenario_ctx["record_forgetting_id"],
        scenario_ctx["record_recent_id"],
    }
    scenario_preview = [item for item in forgetting_preview if item.get("id") in scenario_ids]
    important_results = memory_store.search(
        query="大阪旅行 優先",
        query_vector=None,
        user_id=DEMO_USER_ID,
        agent_id=DEMO_AGENT_ID,
        options=MemorySearchOptions(
            mode=SearchMode.KEYWORD,
            ranking_profile=RankingProfile.IMPORTANT,
            top=10,
            category=scenario_ctx["scenario_tag"],
            include_archived=True,
        ),
    )

    print("[4] 忘却 dry-run と important ranking")
    print(json.dumps(
        {
            "forgetting_preview": scenario_preview,
            "mode": str(SearchMode.KEYWORD),
            "ranking_profile": str(RankingProfile.IMPORTANT),
            "important_results": summarize_scored_results(important_results),
        },
        ensure_ascii=False,
        indent=2,
    ))

### 9-5. 検索可視性を確認する

このセルでは、既定検索と `include_archived=True` を指定した検索を比較します。目的は、記憶の状態が検索結果にどう影響するかを確認することです。

既定の `MemorySearchOptions` では、検索対象は `active` の記憶だけです。`include_archived=True` を指定すると `archived` も候補に入ります。一方で、`deleted` はユーザーが忘れてほしいと示した記憶なので、既定検索にも確認用検索にも出さない設計にしています。

確認ポイント:

- `deleted` の記憶は検索結果に出ない
- `archived` の記憶は、明示的に含めた検索で確認できる
- エージェントに渡す検索と、運用確認用の検索では `MemorySearchOptions` を使い分ける

In [ ]:
if "scenario_ctx" not in globals() or not scenario_ctx:
    print("スキップ: シナリオ初期化セルを先に実行してください。")
else:
    default_results = memory_store.search(
        query="大阪旅行",
        query_vector=memory_intelligence.embed("大阪旅行"),
        user_id=DEMO_USER_ID,
        agent_id=DEMO_AGENT_ID,
        options=MemorySearchOptions(
            mode=SearchMode.HYBRID,
            top=10,
            category=scenario_ctx["scenario_tag"],
        ),
    )
    include_archived_results = memory_store.search(
        query="大阪旅行",
        query_vector=memory_intelligence.embed("大阪旅行"),
        user_id=DEMO_USER_ID,
        agent_id=DEMO_AGENT_ID,
        options=MemorySearchOptions(
            mode=SearchMode.HYBRID,
            top=10,
            category=scenario_ctx["scenario_tag"],
            include_archived=True,
        ),
    )

    print("[5] 検索可視性と hybrid search")
    print(json.dumps(
        {
            "mode": str(SearchMode.HYBRID),
            "default_results": summarize_scored_results(default_results),
            "include_archived_results": summarize_scored_results(include_archived_results),
            "deleted_id": scenario_ctx["record_recall_id"],
            "decayed_id": scenario_ctx["record_forgetting_id"],
            "recent_id": scenario_ctx["record_recent_id"],
        },
        ensure_ascii=False,
        indent=2,
    ))

## 10. Agent Framework での実装例

実アプリでは、作成した `custom_memory_provider` を Agent の `context_providers` に渡します。`before_run()` が Azure AI Search で関連記憶を検索してコンテキストへ注入し、`after_run()` が会話から記憶候補を抽出して保存、更新、削除を担当します。

このセルでは、`agent.run()` の戻り値を表示したうえで、実行後に同じテーマでメモリー検索を行います。OpenTelemetry の `tracer.start_as_current_span(...)` で `agent.run()` と実行後検索を囲むため、Jaeger などの OTLP 送信先でエージェント実行とメモリー処理の流れを確認できます。

In [ ]:
import os
from agent_framework.openai import OpenAIChatCompletionClient

mcp_server_uri = os.getenv("MCP_SERVER_URI")
azure_openai_key = os.getenv("AZURE_OPENAI_API_KEY")
azure_openai_endpoint = os.getenv("AZURE_OPENAI_ENDPOINT")
azure_deployment = os.getenv("AZURE_OPENAI_CHAT_DEPLOYMENT")
api_version = os.getenv("AZURE_OPENAI_API_VERSION")

# === 認証方式の選択 ===
# True: APIキー認証, False: DefaultAzureCredential 認証
USE_KEY_AUTH = os.getenv("USE_KEY_AUTH", "False").lower() in ("true", "1", "t")

if USE_KEY_AUTH:
    auth_kwargs = dict(api_key=azure_openai_key)
    print("🔑 認証方式: APIキー認証")
else:
    from azure.identity.aio import DefaultAzureCredential
    auth_kwargs = dict(credential=DefaultAzureCredential())
    # フレームワークが環境変数の AZURE_OPENAI_API_KEY を自動読み込みして
    # credential より優先してしまうため、明示的に削除する
    os.environ.pop("AZURE_OPENAI_API_KEY", None)
    print("🔐 認証方式: DefaultAzureCredential")

project_endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
model_deployment = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

# NOTE: Agent を `async with` で使うため、client も各実行で新しく生成します。
def create_azure_openai_chat_client():
    return OpenAIChatCompletionClient(
        **auth_kwargs,
        azure_endpoint=azure_openai_endpoint,
        api_version=api_version,
        model=azure_deployment
    )

print(f"MCP Server URI: {mcp_server_uri}")
print(f"Foundry Project Endpoint: {project_endpoint}")
print(f"Foundry Model Deployment: {model_deployment}")
print(f"Azure OpenAI Endpoint: {azure_openai_endpoint}, Deployment: {azure_deployment}")

In [ ]:
print("Agent Framework 配線例")

from agent_framework import Agent


def render_message_text(message: Any) -> str:
    text = getattr(message, "text", None)
    if text:
        return str(text)
    contents = getattr(message, "contents", None)
    if contents:
        return "\n".join(str(item) for item in contents)
    content = getattr(message, "content", None)
    if content:
        return str(content)
    return str(message)


def render_agent_result(result: Any) -> str:
    text = getattr(result, "text", None)
    if text:
        return str(text)
    messages = getattr(result, "messages", None)
    if messages:
        return "\n".join(render_message_text(message) for message in messages)
    value = getattr(result, "value", None)
    if value is not None and value is not result:
        return render_agent_result(value)
    content = getattr(result, "content", None)
    if content:
        return str(content)
    return str(result)


def summarize_agent_memory_results(results: list[MemorySearchResult]) -> list[dict[str, Any]]:
    return [
        {
            "rank": index,
            "score": result.score,
            "reranker_score": result.reranker_score,
            "memory": result.record.memory,
            "category": result.record.category,
            "importance": result.record.importance,
            "retention_score": result.record.retention_score,
            "access_count": result.record.access_count,
            "status": str(result.record.status),
            "updated_at": dt_to_json(result.record.updated_at),
        }
        for index, result in enumerate(results, start=1)
    ]


if custom_memory_provider is None:
    print("スキップ: custom_memory_provider が初期化されていません。")
elif not azure_openai_endpoint or not azure_deployment or not api_version:
    print("スキップ: Azure OpenAI の endpoint、deployment、api_version のいずれかが不足しています。")
else:
    user_query = "大阪旅行の計画を立てて"
    agent = Agent(
        client=create_azure_openai_chat_client(),
        name="CustomMemoryTravelAgent",
        instructions="関連する記憶がある場合だけ利用して回答してください。",
        context_providers=[custom_memory_provider],
    )

    agent_session = agent.create_session()
    with tracer.start_as_current_span("custom_memory_agent_run_plan") as span:
        span.set_attribute("agent.name", "CustomMemoryTravelAgent")
        span.set_attribute("user.query", user_query)
        span.set_attribute("memory.search.mode", str(custom_memory_provider.search_options.mode))
        span.set_attribute("memory.search.top", custom_memory_provider.search_options.top)
        agent_result = await agent.run(user_query, session=agent_session)

    print("\n[1] エージェント応答")
    print(render_agent_result(agent_result))


In [ ]:
if custom_memory_provider is None:
    print("スキップ: custom_memory_provider が初期化されていません。")
elif not azure_openai_endpoint or not azure_deployment or not api_version:
    print("スキップ: Azure OpenAI の endpoint、deployment、api_version のいずれかが不足しています。")
else:
    user_query = "大阪旅行で優先したいものといえば？"
    agent = Agent(
        client=create_azure_openai_chat_client(),
        name="CustomMemoryTravelAgent",
        instructions="関連する記憶がある場合だけ利用して回答してください。",
        context_providers=[custom_memory_provider],
    )

    agent_session = agent.create_session()
    with tracer.start_as_current_span("custom_memory_agent_run_priority_question") as span:
        span.set_attribute("agent.name", "CustomMemoryTravelAgent")
        span.set_attribute("user.query", user_query)
        span.set_attribute("memory.search.mode", str(custom_memory_provider.search_options.mode))
        span.set_attribute("memory.search.top", custom_memory_provider.search_options.top)
        agent_result = await agent.run(user_query, session=agent_session)

    print("\n[1] エージェント応答")
    print(render_agent_result(agent_result))


## 11. まとめ

メモリーのカスタムを行うと次の責務を自分で持つことになります。ただし、検索エンジンでできる順位付けは Python 側に抱え込まず、Azure AI Search の機能として設計します。

| 責務 | このノートブックのクラス | 読みどころ |
|---|---|---|
| 会話から記憶候補を抽出 | `MemoryExtractor` | どの発話を記憶として残すか |
| 既存記憶との統合判断 | `MemoryConsolidator` | 追加、更新、削除、何もしない判断をどう分けるか |
| 埋め込み生成 | `AzureOpenAIMemoryIntelligence.embed()` | Azure OpenAI がない時のフォールバックをどう扱うか |
| 検索、保存、フィルター、状態管理 | `AzureAISearchMemoryStore` | Azure AI Search のフィールドを検索条件へどう変換するか |
| 検索順位の調整 | Azure AI Search のスコアリング プロファイル | `RankingProfile.IMPORTANT` と `RankingProfile.RECENT` が内部で順位付けルールを選ぶ |
| 検索方針の指定 | `MemorySearchOptions` | どの記憶をエージェントに見せるか、どの検索方式と順位付け方針を使うか |
| エージェント実行への注入と保存 | `AzureAISearchMemoryContextProvider` | 実行前に思い出し、実行後に記憶を更新する流れ |
| 忘却、想起、古い記憶の扱い | `HumanMemoryPolicy` | 使われた記憶を強め、古い記憶を弱める考え方 |

この構成の強みは、Azure AI Search のスキーマを記憶用に正面から設計できることです。自由な JSON に閉じ込めず、日付、重要度、状態、アクセス回数、期限をフィールド化することで、フィルター、ソート、スコアリング プロファイル、セマンティック ランカーへ自然に接続できます。

実運用で最初に調整する値は、`MemorySearchOptions` の `mode`、`ranking_profile`、`top`、`min_retention_score`、`include_archived` です。回答に不要な記憶が混ざる場合は件数や保持スコアを絞り、重要度を強く見たい場合は `RankingProfile.IMPORTANT`、最近更新を強く見たい場合は `RankingProfile.RECENT` を選びます。

## 参考資料
### 9-0. 4つのスコア項目はいつ更新されるか

`importance`、`retention_score`、`access_count`、`updated_at` は、どれも検索や記憶の寿命に関わりますが、役割と更新タイミングは違います。特に `updated_at` は「思い出した時刻」ではなく「記憶そのものを追加、更新、削除した時刻」として扱い、想起時刻は `last_accessed_at` に分けています。

| フィールド | 更新タイミング | 計算ロジック | 主な使い道 |
|---|---|---|---|
| `importance` | 記憶候補の抽出時、または既存記憶への統合時 | LLM 抽出では LLM が返した `importance` を `coerce_unit_score()` で 0.0 から 1.0 に正規化します。LLM が使えない rule fallback では、永続的な好みや制約らしい発話を `0.65` として保存します。候補が明示値を持たない場合の既定値は `MemoryCandidate` の `0.5` です。既存記憶を `UPDATE` する場合は `max(existing.importance, candidate.importance)` を使い、重要度を不用意に下げません。 | `retention_score` の土台になり、`RankingProfile.IMPORTANT` の順位調整に使います。 |
| `retention_score` | `ADD`、`UPDATE`、想起時の `touch_results()`、忘却チェックの `apply_forgetting()` | `HumanMemoryPolicy.retention_score()` で計算します。期限切れなら `0.0`。それ以外は `decay * importance_anchor + access_rehearsal` を 0.0 から 1.0 に丸めます。`decay = 0.5 ** (age_days / half_life_days)`、`importance_anchor = 0.35 + 0.65 * importance`、`access_rehearsal = min(access_count * access_boost, 0.35)` です。 | 古く弱い記憶を検索から外す判断と、`RankingProfile.IMPORTANT` の順位調整に使います。 |
| `access_count` | エージェント実行前の記憶検索で実際に使われた時 | `before_run()` が関連記憶を検索し、見つかった結果だけ `store.touch_results()` に渡します。その中で `policy.touch()` が `access_count += 1` します。単にデバッグ用に `search_memories()` を直接呼ぶだけでは増えません。 | よく思い出される記憶を残しやすくし、`RankingProfile.IMPORTANT` で上位に寄せます。 |
| `updated_at` | 記憶を新規追加した時、既存記憶を統合更新した時、削除状態へ変更した時 | `ADD` では `created_at` と `updated_at` を現在時刻にします。`UPDATE` では本文やカテゴリなどを統合したうえで `updated_at = now` にします。`DELETE` では `mark_status()` が `status` と `updated_at` を更新します。想起だけでは `updated_at` は変えず、`last_accessed_at` を更新します。 | `RankingProfile.RECENT` の Freshness スコアリングで、最近変更された記憶を上位に寄せます。 |

実行サイクルで見ると、ユーザー入力の直前に `before_run()` が関連記憶を検索し、使われた記憶だけ `access_count`、`last_accessed_at`、`retention_score` を更新します。エージェント応答後の `after_run()` では、今回の会話から記憶候補を抽出し、`ADD` / `UPDATE` / `DELETE` / `NONE` を判断します。`ADD` と `UPDATE` では `importance` と `retention_score` が確定し、`updated_at` も更新されます。

Azure AI Search 側では、`RankingProfile.IMPORTANT` が `importance`、`retention_score`、`access_count` をまとめてブーストします。`RankingProfile.RECENT` は `updated_at` を強く見ます。つまり Python 側はライフサイクルの値を正しく更新し、検索順位の最終調整は Search のスコアリング プロファイルに任せる設計です。

#### `importance` スコアについて
まず `importance` を決める主体は Azure AI Search ではありません。Search は保存済みの `importance` を順位付けに使うだけです。値を決めるのは Python 側の `MemoryExtractor` で、LLM が使える場合は LLM に記憶候補と一緒に重要度を JSON で返させます。LLM が使えない場合は rule fallback が、好み、優先、苦手、必要条件など永続的な発話らしいものに固定値を付けます。その後、既存記憶へ統合する `UPDATE` では `MemoryConsolidator` の判断結果を受けて、既存値と新候補の高い方を残します。

#### `updated_at`/`retention_score` について
`updated_at` のブーストがあるのに `retention_score` も使う理由は、両者が別の問いに答えるためです。`updated_at` は「最近変更されたか」という鮮度のシグナルです。一方で `retention_score` は「その記憶をまだ残す価値があるか、回答に使ってよいか」という保持価値のシグナルです。たとえば半年前の「ベジタリアン対応を優先する」は古くても重要なので残したい一方、昨日の「明日の会議は10時」は新しくても期限切れなら弱めたい、という差を表現できます。

そのため、`updated_at` だけに頼ると古いが重要な好みを落としすぎたり、最近追加された一時的な予定を強くしすぎたりします。`retention_score` は `importance`、経過時間、想起回数、期限をまとめて評価するので、鮮度だけでは表せない「記憶としての寿命」を Search の順位とフィルターに渡せます。


## 人間らしい記憶: HumanMemoryPolicy

エージェントの記憶が増え続けると、古い予定や弱い好みまで毎回参照してしまい応答品質を下げます。`HumanMemoryPolicy` は、思い出された記憶を残しやすくし、使われない記憶を徐々に見えにくくする仕組みです。

### 保持スコアの計算式（簡易的）
認知科学的にはもっと究めることができるようですが、今回デモ用途として簡易的に実装しています。
保持スコアは「この記憶をまだエージェントに見せる価値があるか」を 0.0〜1.0 で表す数値です。スコアが下がると検索フィルターで除外され、閾値を下回ると `archived` 状態に遷移します。

式は 3 つの独立した要素の組み合わせです。

$$
\text{retention\_score} = \underbrace{\text{decay} \times \text{importance\_anchor}}_{\text{時間と重要度による基本値}} + \underbrace{\text{access\_rehearsal}}_{\text{想起による底上げ}}
$$

前半の積が「時間経過と重要度で決まる基本的な残り具合」を表し、後半の加算項が「実際に使われた実績による底上げ」を表します。

---

#### 要素 1: decay（時間減衰）

$$
\text{decay} = 0.5^{\frac{\text{age\_days}}{\text{half\_life\_days}}}
$$

decay は、記憶が最後に更新されてから何日経過したかに基づいて、忘れやすさを 0.0〜1.0 で表します。放射性元素の半減期と同じ考え方で、既定の半減期は 45 日です。つまり、45 日経つとスコアが半分になり、90 日で 4 分の 1、180 日で約 0.06 まで落ちます。作成直後は 1.0 で、時間が経つにつれ指数的に減少します。

この要素だけでは、ユーザーの長期的な好み（「ベジタリアン対応を優先」）も、一時的なメモ（「明日の会議は 10 時」）も同じペースで消えてしまいます。それを補正するのが次の要素です。

---

#### 要素 2: importance_anchor（重要度による減衰の下支え）

$$
\text{importance\_anchor} = 0.35 + 0.65 \times \text{importance}
$$

importance_anchor は decay に掛ける係数で、重要度が高い記憶ほど同じ日数が経過しても高い値を維持するための仕組みです。

具体的に言うと、重要度 1.0 の記憶ではアンカーが 1.0 になるため、decay がそのまま基本値になります。一方、重要度 0.0 の記憶ではアンカーが 0.35 に下がるため、decay が同じでも基本値は 3 分の 1 程度しか残りません。たとえば 90 日後の decay は 0.25 ですが、重要度 1.0 なら基本値は 0.25 × 1.0 = 0.25 で閾値 0.22 をまだ超えています。重要度 0.0 なら 0.25 × 0.35 = 0.088 で、とっくにアーカイブ対象です。

下限を 0.35 にしている理由は、重要度がゼロの記憶であっても作成直後の数十日間は検索結果に残るようにするためです。記憶を作った直後にいきなり消えてしまっては実用になりません。

---

#### 要素 3: access_rehearsal（想起による底上げ）

$$
\text{access\_rehearsal} = \min(\text{access\_count} \times 0.04,\ 0.35)
$$

access_rehearsal は、エージェントの実行時に検索結果として実際に使われた回数に応じて、スコア全体に加算される底上げ値です。1 回使われるごとに 0.04 ずつ増え、上限は 0.35 です。

この要素は前半の積（decay × anchor）とは独立に加算されます。そのため、decay がほぼゼロになった古い記憶でも、9 回以上使われていれば rehearsal だけで 0.35 に達し、アーカイブ閾値の 0.22 を上回り続けます。言い換えると、**頻繁に思い出される記憶は、どれだけ古くてもエージェントの視界に残り続ける**設計です。

これは人間が「毎日使う住所は何十年経っても忘れない」のと同じ直感に基づいています。逆に、一度も使われなかった記憶は rehearsal が 0 のままなので、decay と anchor だけでスコアが決まり、時間とともに自然に見えなくなります。

---

#### 3 つの要素がどう組み合わさるか

式の全体を改めて読むと、次のような意味になります。

1. まず、記憶が作られてからの経過日数で基本的な「新鮮さ」が決まります（decay）。
2. その新鮮さに重要度の重みを掛けて、「時間と重要度を踏まえた基本値」が出ます（decay × anchor）。
3. 最後に、実際に使われた実績分を足して最終スコアにします（+ rehearsal）。

この構造により、3 種類の記憶がそれぞれ違う振る舞いをします。

**新しくて重要な記憶**は decay が高く anchor も高いため、基本値だけでスコアが高くなります。rehearsal はまだゼロでも問題ありません。

**古いが頻繁に使われる記憶**は decay が低いため基本値は小さくなりますが、rehearsal の底上げによって閾値を超え続けます。たとえば半年前に作られた重要度 1.0 の記憶は、decay × anchor = 0.06 × 1.0 = 0.06 まで落ちますが、9 回使われていれば 0.06 + 0.35 = 0.41 で十分に active です。

**古くて使われない記憶**は decay が下がり、rehearsal もゼロのままなので、自然にアーカイブ閾値を割り込みます。重要度が低ければさらに早く消えます。これが「使われない記憶はフェードアウトする」挙動を生みます。

---

#### パラメーター一覧

| パラメーター | 既定値 | 意味 |
|---|---|---|
| `half_life_days` | 45.0 | 時間減衰の半減期。大きくすると記憶全体が長持ちする |
| `access_boost` | 0.04 | 1 回想起ごとの強化量（最大 0.35 で飽和） |
| `archive_threshold` | 0.22 | これ以下で `archived` へ遷移し、既定検索から除外される |

### 想起による強化

`before_run()` の検索で使われた記憶は `touch_results()` により以下が更新されます。

```python
record.last_accessed_at = now
record.access_count += 1
record.retention_score = policy.retention_score(record)
record.status = policy.status_for(record)
```

**よく使う記憶は残りやすく、使わない記憶は自然にフェードアウトする**。Ebbinghaus の忘却曲線を簡略化したモデルです。

### 4 つのスコア項目の更新タイミング

| フィールド | 更新されるタイミング | 検索への影響 |
|---|---|---|
| `importance` | 記憶候補の抽出時、既存記憶への統合時 | `RankingProfile.IMPORTANT` の順位調整 |
| `retention_score` | ADD / UPDATE / 想起時 / 忘却チェック時 | フィルター除外 + `RankingProfile.IMPORTANT` の順位調整 |
| `access_count` | `before_run()` の検索結果が `touch_results()` された時 | `RankingProfile.IMPORTANT` の順位調整 |
| `updated_at` | ADD / UPDATE / DELETE 時（想起では更新しない） | `RankingProfile.RECENT` の Freshness スコア |
